# Recommender Systems — Week 2 Study Notebook
### Baselines → Neighborhood CF (kNN) → Matrix Factorization & LightGCN

این نوت‌بوک سه نوت‌بوک استاد (`baselines.ipynb`، `knn_cf.ipynb`، `gnn.ipynb`) را در **یک خط داستانی واحد** بازنویسی می‌کند. هر بخش دقیقاً همین ترتیب را دارد:

| گام | چه چیزی | هدف |
|---|---|---|
| A | **Problem** — مسئله از کجا می‌آید و کِی از این روش استفاده می‌کنیم | درک صورت مسئله |
| B | **Algorithm** — فرمول، pseudocode، مثال دستی، complexity | امتحان |
| C | **Professor's code** — کد استاد خط‌به‌خط + ایرادهایش | امتحان + فهم کد |
| D | **Industry version** — همان کار با کتابخانه‌هایی که واقعاً استفاده می‌شوند | دید صنعتی |
| E | **Exam** — سوال‌های استاد + سوال‌های محتمل با جواب نمونه (انگلیسی) | امتحان |

**زبان:** توضیح‌ها فارسی، اصطلاحات و جواب‌های نمونه‌ی امتحان انگلیسی (چون سر امتحان باید انگلیسی بنویسی).

---
## 0. The big picture — نقشه‌ی کل هفته

یک recommender system فقط یک ورودی اصلی دارد: ماتریس تعامل **user × item** که تقریباً همه‌اش خالی است (sparse). کل درس جواب دادن به این سوال است: **خانه‌های خالی را چطور پر کنیم؟**

دو نوع task داریم و این تفکیک در همه‌ی سوال‌های امتحان مهم است:

| | Rating Prediction | Top-N Recommendation |
|---|---|---|
| سوال | کاربر $u$ به آیتم $i$ چه نمره‌ای می‌دهد؟ | کدام $N$ آیتم را به $u$ نشان بدهم؟ |
| نوع مسئله | regression | ranking |
| داده | explicit (ستاره) | explicit یا implicit (click, view, purchase) |
| متریک | MAE, RMSE | Hit Ratio, Precision@K, Recall@K, NDCG@K |
| در صنعت | کم‌کاربرد (میراث Netflix Prize) | **تقریباً همه‌ی سیستم‌های واقعی** |

خط سیر هفته:

1. **Baselines** — ساده‌ترین جواب‌های ممکن. هر مدلی باید این‌ها را شکست بدهد.
2. **Neighborhood CF (kNN)** — «آیتم‌های شبیه به چیزهایی که دوست داشتی». اولین روش personalized.
3. **MF و LightGCN** — به‌جای شمردن شباهت، برای هر user و item یک embedding یاد می‌گیریم.

هر مرحله ضعف مرحله‌ی قبل را حل می‌کند: baseline شخصی‌سازی ندارد → kNN شخصی‌سازی دارد ولی $O(n^2)$ است و با sparsity ضعیف می‌شود → MF فشرده و سریع است ولی فقط تعامل مستقیم را می‌بیند → LightGCN همسایه‌های چندقدمی گراف را هم وارد embedding می‌کند.

### Setup

In [1]:
# Colab: این خط را یک بار اجرا کن (روی Colab حدود ۱ دقیقه)
# %pip install -q scikit-surprise implicit

In [2]:
import os, math, time, random, heapq, bisect, itertools, operator, zipfile, urllib.request, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 4)
SEED = 42
np.random.seed(SEED); random.seed(SEED)

In [3]:
# MovieLens ml-latest-small: همان دیتاست استاد
if not os.path.exists("ml-latest-small/ratings.csv"):
    urllib.request.urlretrieve("https://files.grouplens.org/datasets/movielens/ml-latest-small.zip", "ml.zip")
    zipfile.ZipFile("ml.zip").extractall()

ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies  = pd.read_csv("ml-latest-small/movies.csv")
title   = movies.set_index("movieId").title

n_users, n_items, n_ratings = ratings.userId.nunique(), ratings.movieId.nunique(), len(ratings)
print(f"users={n_users}  items={n_items}  ratings={n_ratings}")
print(f"density = {n_ratings/(n_users*n_items):.4f}   -> {100*(1-n_ratings/(n_users*n_items)):.1f}% of the matrix is empty")
ratings.head(3)

users=610  items=9724  ratings=100836
density = 0.0170   -> 98.3% of the matrix is empty


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224


**نکته‌ی امتحانی:** استاد در کدش `n_ratings/(n_users*n_items)` را با اسم *Sparsity* چاپ می‌کند. این عدد در واقع **density** است؛ `sparsity = 1 − density`. اگر پرسیدند «sparsity این دیتاست چقدر است» جواب ≈ 98.3٪ است، نه 1.7٪.

دو واقعیت دیگر درباره‌ی داده که در ادامه مدام به آن برمی‌گردیم:

In [4]:
item_cnt = ratings.groupby("movieId").size().sort_values(ascending=False)
print("rating distribution:\n", ratings.rating.value_counts(normalize=True).sort_index().round(3).to_string())
print(f"\nitems with <=2 ratings : {100*(item_cnt<=2).mean():.1f}% of the catalog   (long tail)")
print(f"top 10% of items hold  : {100*item_cnt.head(n_items//10).sum()/n_ratings:.1f}% of all ratings (popularity bias)")

rating distribution:
 rating
0.5    0.014
1.0    0.028
1.5    0.018
2.0    0.075
2.5    0.055
3.0    0.199
3.5    0.130
4.0    0.266
4.5    0.085
5.0    0.131

items with <=2 ratings : 48.8% of the catalog   (long tail)
top 10% of items hold  : 60.0% of all ratings (popularity bias)


---
# Part 1 — Baselines

## 1.A Problem — چرا baseline؟

قبل از ساختن هر مدلی باید بدانی «بدون هیچ هوشمندی چه عددی می‌گیرم». بدون baseline عدد MAE = 0.75 هیچ معنایی ندارد.

**کِی در عمل از baseline استفاده می‌شود:**

- **Sanity check / lower bound** — مدل پیچیده‌ای که popularity را نبرد، ارزش deploy ندارد.
- **Cold start** — کاربر تازه‌وارد هیچ تاریخچه‌ای ندارد، پس «Trending / Most popular» تنها گزینه است. صفحه‌ی اول Netflix و YouTube برای کاربر logout همین است.
- **Fallback** — وقتی مدل اصلی برای یک user/item جواب ندارد (استاد هم در کدش به `avg_rating` fallback می‌کند).
- **Feature** — در سیستم‌های واقعی popularity و میانگین‌ها به‌عنوان feature وارد ranker می‌شوند.

## 1.B Algorithms

### Rating prediction baselines

| روش | فرمول | شخصی‌سازی؟ |
|---|---|---|
| Random | $\hat r_{ui} \sim \text{Uniform}\{1..5\}$ | ✗ |
| Global mean | $\hat r_{ui} = \mu$ | ✗ |
| Item mean | $\hat r_{ui} = \bar r_i$ | فقط سمت item |
| User mean | $\hat r_{ui} = \bar r_u$ | فقط سمت user |
| **Bias baseline** | $\hat r_{ui} = \mu + b_u + b_i$ | هر دو |

مورد آخر در جزوه‌ی استاد نیست ولی **ادامه‌ی طبیعی** همان ایده است و در صنعت «baseline استاندارد» همین است: بعضی کاربرها سخت‌گیرند ($b_u<0$)، بعضی فیلم‌ها ذاتاً خوب‌اند ($b_i>0$).

$$b_i = \frac{\sum_{u \in U_i}(r_{ui}-\mu)}{\lambda_i + |U_i|}, \qquad b_u = \frac{\sum_{i \in I_u}(r_{ui}-\mu-b_i)}{\lambda_u + |I_u|}$$

$\lambda$ همان **damping / shrinkage** است: فیلمی که فقط ۱ رأی ۵ ستاره دارد نباید $b_i$ بزرگی بگیرد؛ $\lambda$ آن را به سمت صفر (یعنی به سمت $\mu$) می‌کشد.

### Metrics

$$\text{MAE} = \frac{1}{|T|}\sum_{(u,i)\in T} |r_{ui}-\hat r_{ui}| \qquad \text{RMSE} = \sqrt{\frac{1}{|T|}\sum_{(u,i)\in T} (r_{ui}-\hat r_{ui})^2}$$

RMSE خطاهای بزرگ را بیشتر جریمه می‌کند (به‌خاطر توان ۲)، پس همیشه $\text{RMSE} \ge \text{MAE}$. **مثال دستی:** خطاها $[0, 1, 3]$ → MAE $= 4/3 = 1.33$، RMSE $= \sqrt{10/3} = 1.83$.

### Top-N baselines

| روش | score هر آیتم | ضعف |
|---|---|---|
| Random | عدد تصادفی | lower bound |
| **Popularity** | تعداد rating ها (count) | برای همه یکسان، بدون novelty |
| Highest average | میانگین rating | آیتم با ۱ رأیِ ۵ ستاره اول می‌شود |
| Damped average | $\dfrac{\sum r + \lambda\mu}{n+\lambda}$ | راه‌حلِ مشکل بالا (IMDb weighted rating همین است) |

### Hit Ratio با پروتکل «1 positive + 99 negatives»

```
for each user u:
    pos  <- one item u rated 5                       # ground truth
    negs <- 99 random items u has NOT rated
    rank the 100 candidates by the model's score
    hit@K <- 1 if pos is in the top K else 0
HR@K = (#hits) / (#users evaluated)
```

**چرا 99 منفی و نه کل catalog؟** رتبه‌بندی کل catalog برای هر کاربر گران است؛ sampling ارزان‌ترش می‌کند (این پروتکل از مقاله‌ی NCF رایج شد).

**عدد مرجع که باید حفظ باشی:** برای random، مثبت در هر کدام از ۱۰۰ جایگاه با احتمال برابر می‌افتد، پس

$$\mathbb{E}[\text{HR@}K]_{\text{random}} = \frac{K}{100} \;\Rightarrow\; \text{HR@5}=0.05,\; \text{HR@10}=0.10,\; \text{HR@20}=0.20$$

**Complexity:** ساختن جدول میانگین/popularity یک pass روی داده است: $O(|R|)$. هر پیش‌بینی $O(1)$. برای همین baseline ها عملاً رایگان‌اند.

## 1.C Professor's code — چه کار می‌کند

ساختار کد استاد در `baselines.ipynb`:

1. **Random prediction** — روی *همه‌ی* rating ها یک عدد تصادفی `np.random.randint(1,6)` می‌سازد و MAE/RMSE می‌گیرد (train/test لازم ندارد چون چیزی یاد نمی‌گیرد).
2. **Average prediction** — `train_test_split(test_size=0.1)`؛ میانگین‌ها **فقط از train** حساب می‌شوند؛ با `groupby('movieId')['rating'].mean()` جدول item-average و user-average می‌سازد؛ در test با `.get(movie_id, avg_rating)` اگر آیتم در train نبود به global mean برمی‌گردد.
3. **Top-N** — همان حلقه‌ی pseudocode بالا، سه بار با سه score مختلف (popularity، random، average rating).

نسخه‌ی فشرده‌ی همان منطق (با حلقه، به سبک استاد) تا ببینی دقیقاً چه اتفاقی می‌افتد:

In [5]:
X_train, X_test = train_test_split(ratings, test_size=0.1, random_state=SEED)

avg_rating    = X_train.rating.mean()                       # global mean (only from train!)
item_averages = X_train.groupby("movieId")["rating"].mean() # Series: movieId -> mean
user_averages = X_train.groupby("userId")["rating"].mean()

err = 0
for _, row in X_test.iterrows():                            # professor-style explicit loop
    pred = item_averages.get(row["movieId"], avg_rating)    # fallback for unseen items
    err += abs(row["rating"] - pred)
print("Item-average MAE (loop version):", round(err/len(X_test), 4))

Item-average MAE (loop version): 0.761


**ایرادها و نکته‌های کد استاد** (این‌ها دقیقاً چیزهایی‌اند که سوال مفهومی از آن‌ها درمی‌آید):

1. **`.iloc` و `.iterrows()` داخل حلقه** کند است (هر بار یک Series می‌سازد). در عمل vectorized می‌نویسیم (بخش 1.D).
2. **Random روی کل داده، بقیه روی test** — مقایسه‌ی کاملاً یکسانی نیست، ولی برای random مهم نیست.
3. **Rating های واقعی half-star دارند** (0.5 تا 5) ولی random فقط عدد صحیح 1..5 می‌سازد.
4. **Data leakage در Top-N:** آیتم مثبت از کل `ratings` انتخاب می‌شود ولی `item_popularity` از `X_train` (۹۰٪ داده) حساب شده؛ یعنی همان rating ای که داریم تست می‌کنیم معمولاً در popularity شمرده شده است. اصولاً باید positive از train حذف شود.
5. **چرا HR@20 برای popularity بالای 0.9 می‌شود؟** این بیشتر خاصیت **پروتکل** است تا قدرت popularity: آیتم مثبت چیزی است که کاربر واقعاً دیده (پس ذاتاً popular است)، ولی ۹۹ منفی uniform از کل catalog می‌آیند که بیشترشان long-tail با ۱ یا ۲ رأی‌اند. پایین همین را با آزمایش نشان می‌دهم.

## 1.D Industry version

### Rating prediction — vectorized pandas

همان محاسبات، بدون حلقه. الگوی `map` + `fillna` معادل `.get(key, default)` استاد است.

In [6]:
def mae(y, p):  return float(np.mean(np.abs(y - p)))
def rmse(y, p): return float(np.sqrt(np.mean((y - p) ** 2)))

def fit_bias_baseline(train, lam_i=10, lam_u=5):
    """r_hat = mu + b_u + b_i  with damped (shrunk) biases."""
    mu  = train.rating.mean()
    d   = train.assign(res=train.rating - mu)
    g   = d.groupby("movieId").res
    b_i = g.sum() / (lam_i + g.count())
    d["res2"] = d.res - d.movieId.map(b_i)                 # remove item effect before fitting user effect
    g   = d.groupby("userId").res2
    b_u = g.sum() / (lam_u + g.count())
    return mu, b_u, b_i

y  = X_test.rating.to_numpy()
mu = avg_rating
mu_, b_u, b_i = fit_bias_baseline(X_train)

preds = {
    "Random (1..5)":      np.random.default_rng(SEED).integers(1, 6, len(X_test)),
    "Global mean":        np.full(len(X_test), mu),
    "Item mean":          X_test.movieId.map(item_averages).fillna(mu).to_numpy(),
    "User mean":          X_test.userId.map(user_averages).fillna(mu).to_numpy(),
    "Bias: mu+b_u+b_i":   np.clip(mu_ + X_test.userId.map(b_u).fillna(0) + X_test.movieId.map(b_i).fillna(0), 0.5, 5).to_numpy(),
}
rating_table = pd.DataFrame({k: {"MAE": mae(y, p), "RMSE": rmse(y, p)} for k, p in preds.items()}).T
rating_table

,MAE,RMSE
Random (1..5),1.4787,1.8267
Global mean,0.8351,1.0506
Item mean,0.7610,0.9825
User mean,0.7432,0.9563
Bias: mu+b_u+b_i,0.6796,0.8861


**چیزی که باید از جدول بگیری:** هر قدم که اطلاعات بیشتری وارد می‌کنیم (هیچ → میانگین کل → میانگین user/item → هر دو) خطا کم می‌شود. Bias baseline با سه خط کد از هر دو میانگین بهتر است و هر مدل CF باید **این** را شکست بدهد، نه global mean را.

### همین کار با Surprise

`scikit-surprise` کتابخانه‌ی استاندارد برای **rating prediction** است (آموزش، پژوهش، prototype). `NormalPredictor` = random، `BaselineOnly` = همان $\mu+b_u+b_i$، و cross-validation آماده دارد. (`NormalPredictor` از توزیع نرمالِ fit‌شده روی داده نمونه می‌گیرد، نه uniform؛ برای همین خطایش از random ما کمتر است.)

In [7]:
from surprise import Dataset, Reader, NormalPredictor, BaselineOnly
from surprise.model_selection import cross_validate

sdata = Dataset.load_from_df(ratings[["userId", "movieId", "rating"]], Reader(rating_scale=(0.5, 5)))
for name, algo in [("NormalPredictor (random)", NormalPredictor()), ("BaselineOnly (mu+b_u+b_i)", BaselineOnly(verbose=False))]:
    cv = cross_validate(algo, sdata, measures=["MAE", "RMSE"], cv=3, verbose=False)
    print(f"{name:28s} MAE={cv['test_mae'].mean():.4f}  RMSE={cv['test_rmse'].mean():.4f}")

NormalPredictor (random)     MAE=1.1410  RMSE=1.4304


BaselineOnly (mu+b_u+b_i)    MAE=0.6769  RMSE=0.8773


### Top-N baselines — پروتکل استاد، بدون leakage

تفاوت با کد استاد: (۱) آیتم مثبت از train **حذف** می‌شود (leave-one-out)، (۲) یک تابع evaluation برای همه‌ی روش‌ها، (۳) tie ها تصادفی شکسته می‌شوند.

In [8]:
def leave_one_out(ratings, pos_thr=5.0, seed=SEED):
    """Hold out ONE positive (rating >= pos_thr) per user as test; everything else is train."""
    test  = ratings[ratings.rating >= pos_thr].groupby("userId").sample(n=1, random_state=seed)
    train = ratings.drop(test.index)
    return train, test

all_items  = ratings.movieId.unique()
seen_items = ratings.groupby("userId").movieId.apply(set).to_dict()

def sampled_hit_ratio(score_fn, test, Ks=(5, 10, 20), n_neg=99, neg_p=None, seed=SEED):
    """score_fn(array of movieIds) -> array of scores.  Protocol: 1 positive + n_neg sampled negatives."""
    rng, hits = np.random.default_rng(seed), dict.fromkeys(Ks, 0)
    for u, pos in zip(test.userId, test.movieId):
        negs = []
        while len(negs) < n_neg:                                   # sample items the user has not rated
            negs += [m for m in rng.choice(all_items, 2 * n_neg, p=neg_p) if m not in seen_items[u]]
        cand = np.array([pos] + negs[:n_neg])
        s    = score_fn(cand) + rng.random(len(cand)) * 1e-9       # random tie-breaking
        rank = int((s > s[0]).sum())                               # how many candidates beat the positive
        for k in Ks: hits[k] += rank < k
    return {f"HR@{k}": hits[k] / len(test) for k in Ks}

tr, te = leave_one_out(ratings)
mu_tr  = tr.rating.mean()
g      = tr.groupby("movieId").rating
pop, avg = g.count(), g.mean()
LAM = 20
damped = (g.sum() + LAM * mu_tr) / (g.count() + LAM)               # Bayesian / damped average

scorers = {
    "Random":           lambda c: np.zeros(len(c)),
    "Highest average":  lambda c: avg.reindex(c).fillna(mu_tr).to_numpy(),
    "Damped average":   lambda c: damped.reindex(c).fillna(mu_tr).to_numpy(),
    "Popularity":       lambda c: pop.reindex(c).fillna(0).to_numpy(),
}
topn_table = pd.DataFrame({k: sampled_hit_ratio(f, te) for k, f in scorers.items()}).T
print("users evaluated:", len(te))
topn_table

users evaluated: 573


,HR@5,HR@10,HR@20
Random,0.0628,0.1134,0.2269
Highest average,0.0209,0.1449,0.3997
Damped average,0.5672,0.6579,0.7347
Popularity,0.6754,0.7853,0.8761


**تفسیر (همان نتیجه‌ی استاد، با دلیل):**

- **Random** ≈ $K/100$ — نزدیک به عدد تئوری (اختلاف کوچک به‌خاطر تعداد کم کاربران است).
- **Highest average ضعیف است** (در HR@5 حتی از random هم بدتر) چون آیتم‌های niche با ۱ رأی ۵ ستاره بالای لیست می‌نشینند. **Damped average** همین مشکل را حل می‌کند: به محض این‌که تعداد رأی را وارد می‌کنیم عدد جهش می‌کند.
- **Popularity قوی‌ترین است.**

حالا ادعای بخش 1.C را تست کنیم: اگر منفی‌ها را به‌جای uniform، **متناسب با popularity** نمونه بگیریم (یعنی منفی‌های سخت‌تر)، popularity چقدر افت می‌کند؟

In [9]:
p_pop = pop.reindex(all_items).fillna(0).to_numpy(); p_pop = p_pop / p_pop.sum()
pd.DataFrame({
    "Popularity | uniform negatives":            sampled_hit_ratio(scorers["Popularity"], te),
    "Popularity | popularity-sampled negatives": sampled_hit_ratio(scorers["Popularity"], te, neg_p=p_pop),
}).T

,HR@5,HR@10,HR@20
Popularity | uniform negatives,0.6754,0.7853,0.8761
Popularity | popularity-sampled negatives,0.2182,0.3386,0.4747


**درس صنعتی:** عدد HR به‌شدت به **نحوه‌ی انتخاب negative ها** بستگی دارد. با منفی‌های uniform، popularity تقریباً شکست‌ناپذیر به نظر می‌رسد؛ با منفی‌های سخت‌تر عددش به کمتر از نصف می‌رسد. برای همین در صنعت:

- offline metric را روی **full ranking** یا با negative های popularity-aware می‌گیرند (در Part 2 و 3 همین کار را می‌کنیم)،
- کنار accuracy حتماً **coverage / novelty / diversity** را هم گزارش می‌کنند (popularity به همه یک لیست می‌دهد → coverage نزدیک صفر)،
- و تصمیم نهایی با **A/B test آنلاین** است، نه offline metric.

## 1.E Exam — Baselines

> جواب‌ها انگلیسی‌اند تا مستقیم قابل استفاده باشند. سوال‌های «استاد» آن‌هایی‌اند که در متن نوت‌بوک آمده؛ بقیه پیش‌بینی من از شکل‌های دیگرِ همین مطلب است.

**Q (from the notebook's conclusion). Why is popularity such a strong Top-N baseline, and why is "highest average rating" weak?**

*Popularity:* Interaction data follows a long-tail distribution — a small set of items receives most interactions, so a randomly held-out positive item is very likely to be a popular one. Under the 1-positive + 99-random-negatives protocol the negatives are mostly unpopular long-tail items, which makes ranking by count very effective.
*Highest average:* The mean of very few ratings is unreliable. An item with a single 5-star rating gets the maximum score and outranks broadly liked items with hundreds of ratings. The fix is to account for support: require a minimum number of ratings or use a damped (Bayesian) average $\frac{\sum r + \lambda\mu}{n+\lambda}$.

**Q. Why do we need baselines at all?**
They give a minimum performance threshold. A complex model is only justified if it clearly beats cheap heuristics; baselines also serve as cold-start and fallback recommenders in production.

**Q. What HR@10 do you expect from a random recommender under the 1+99 protocol? Why?**
$10/100 = 0.10$. The positive item is equally likely to land in any of the 100 positions, so the probability of being in the top $K$ is $K/100$.

**Q. MAE vs RMSE — which one is larger and what is the difference?**
RMSE ≥ MAE always. RMSE squares the errors, so it penalizes large errors more heavily; MAE weights all errors linearly and is more robust to outliers.

**Q (calculation). True ratings $[4, 2, 5, 3]$, predictions $[3.5, 3.5, 3.5, 3.5]$. Compute MAE and RMSE.**
Errors: $0.5, 1.5, 1.5, 0.5$ → MAE $= 4/4 = 1.0$. Squared: $0.25, 2.25, 2.25, 0.25$ → mean $=1.25$ → RMSE $=\sqrt{1.25}\approx 1.118$.

**Q. Why are the averages computed on the training set only?**
Computing them on all data leaks test ratings into the model (data leakage) and gives an optimistically biased error estimate.

**Q. What happens for a user/item in the test set that never appears in training?**
There is no user/item average — this is the cold-start case. The code falls back to the global average (`.get(id, avg_rating)`).

**Q. What is the main weakness of the popularity recommender even though its hit ratio is high?**
It is not personalized: every user receives the same list. It has near-zero catalog coverage and novelty, and it reinforces popularity bias (rich-get-richer feedback loop).

**Q (algorithm). Write the pseudocode for evaluating a recommender with Hit Ratio@K.** → pseudocode بخش 1.B را بنویس؛ سه نکته‌ای که نباید جا بیفتد: negatives از آیتم‌های **rate نشده** انتخاب می‌شوند، positive باید از train **حذف** شده باشد، و مخرج **تعداد test case ها** است.

---
# Part 2 — Neighborhood-based Collaborative Filtering (kNN)

## 2.A Problem — مسئله از کجا می‌آید

Baseline ها شخصی‌سازی ندارند. **Collaborative Filtering** یعنی فقط از خودِ ماتریس تعامل استفاده کنیم (بدون هیچ اطلاعات محتوایی مثل ژانر یا متن) با این فرض:

> کاربرانی که در گذشته سلیقه‌ی مشابه داشته‌اند، در آینده هم خواهند داشت.

دو نسخه دارد:

| | User-based | Item-based (نسخه‌ی استاد) |
|---|---|---|
| ایده | کاربرهای شبیه تو چه دوست داشتند؟ | چه آیتم‌هایی شبیه چیزهایی‌اند که دوست داشتی؟ |
| شباهت بین | دو سطر ماتریس | دو ستون ماتریس |
| مزیت | serendipity بیشتر | شباهت آیتم‌ها **پایدار** است → offline precompute؛ توضیح‌پذیر («چون X را دیدی») |

**کِی از kNN-CF استفاده می‌کنیم:**

- داده‌ی تعامل کافی داریم ولی content نداریم یا نمی‌خواهیم استفاده کنیم.
- **توضیح‌پذیری** مهم است: «Customers who bought this also bought…» (Amazon، 2003) دقیقاً item-based CF است.
- یک روش قوی، ساده و بدون training می‌خواهیم؛ item-kNN هنوز در benchmark ها رقیب جدی مدل‌های deep است.
- به‌عنوان **candidate generator** در سیستم‌های دو مرحله‌ای (retrieval → ranking).

**کِی جواب نمی‌دهد:** cold start (آیتم/کاربر بدون تعامل)، داده‌ی خیلی sparse (هم‌پوشانی کم → شباهت نویزی)، و scale خیلی بزرگ بدون تقریب ($O(n^2)$).

## 2.B Algorithm

### گام ۱ — Similarity بین دو آیتم

هر آیتم = یک بردار به طول تعداد کاربران (ستون ماتریس). سه معیار:

**Jaccard** — فقط «چه کسی rate کرده»، مقدار rating را نادیده می‌گیرد:
$$J(i,j) = \frac{|U_i \cap U_j|}{|U_i \cup U_j|}$$

**Cosine** — زاویه‌ی بین دو بردار rating:
$$\cos(i,j) = \frac{\sum_{u} r_{ui}\, r_{uj}}{\sqrt{\sum_{u} r_{ui}^2}\;\sqrt{\sum_{u} r_{uj}^2}}$$

**Adjusted cosine / Pearson** — اول mean-center، بعد cosine:
$$\text{sim}(i,j) = \frac{\sum_{u}(r_{ui}-\bar r_u)(r_{uj}-\bar r_u)}{\sqrt{\sum_{u}(r_{ui}-\bar r_u)^2}\;\sqrt{\sum_{u}(r_{uj}-\bar r_u)^2}}$$

**یک انتخاب پنهان در cosine که خیلی مهم است:** جمع‌ها روی کدام کاربرها؟

- **(a) فقط کاربران مشترک** $U_i \cap U_j$ — نسخه‌ی کد استاد (`np.intersect1d`).
- **(b) همه‌ی کاربران، با missing = 0** — نسخه‌ای که کتابخانه‌ها با sparse matrix حساب می‌کنند.

در (b) مخرج norm کل بردار است، پس آیتمی که هم‌پوشانی کمی دارد خودبه‌خود جریمه می‌شود. در (a) اگر فقط **یک** کاربر مشترک باشد، cosine دقیقاً **1.0** می‌شود، مهم نیست چه نمره‌ای داده‌اند. این نکته کلید جواب سوال «Cosine vs Jaccard» استاد است.

**جدول شهودی cosine** (خلاصه‌ی سه cell توضیحی استاد):

| داده | رفتار cosine خام | نتیجه |
|---|---|---|
| mean-centered (مثبت و منفی) | هم‌علامت → +، مخالف → − | agreement واقعی در برابر disagreement |
| ratings مثبت 1..5 | همه‌ی حاصل‌ضرب‌ها مثبت؛ $5\times1=5 > 1\times1=1$ | disagreement را از agreement ضعیف تشخیص نمی‌دهد → برای explicit مناسب نیست |
| implicit 0/1 | $1\times1=1$، بقیه صفر | co-occurrence را می‌شمارد → کاملاً مناسب |

بازه‌ها: Jaccard ∈ [0, 1]؛ cosine روی داده‌ی نامنفی ∈ [0, 1]؛ adjusted cosine / Pearson ∈ [−1, 1].

### مثال دستی (ماتریس تمرین Q4 استاد)

| | A | B | C | D |
|---|---|---|---|---|
| **U1** | 5 | 3 | ? | 1 |
| **U2** | 4 | ? | 4 | 2 |
| **U3** | 1 | 1 | ? | 5 |
| **U4** | 4 | 2 | 3 | ? |

برای A و D: $U_A=\{1,2,3,4\}$، $U_D=\{1,2,3\}$.

- **Jaccard** $= 3/4 = 0.75$
- **Cosine (co-rated)**: $\vec A=[5,4,1]$، $\vec D=[1,2,5]$ → $\dfrac{5+8+5}{\sqrt{42}\sqrt{30}} = \dfrac{18}{35.50} = 0.507$
- **Adjusted cosine**: میانگین کاربران $\bar r_1=3,\ \bar r_2=3.33,\ \bar r_3=2.33$ → $\vec A'=[2,\ 0.67,\ -1.33]$، $\vec D'=[-2,\ -1.33,\ 2.67]$ → $\dfrac{-8.44}{2.49 \times 3.59} = -0.943$

**این مثال را خوب نگاه کن:** A و D در واقع **ضد هم**‌اند (هر که A را دوست دارد D را دوست ندارد). Jaccard می‌گوید 0.75 و cosine خام می‌گوید 0.51 (هر دو «شبیه»!). فقط نسخه‌ی mean-centered حقیقت را می‌گوید: −0.94.

In [10]:
M = pd.DataFrame([[5, 3, np.nan, 1], [4, np.nan, 4, 2], [1, 1, np.nan, 5], [4, 2, 3, np.nan]],
                 index=["U1", "U2", "U3", "U4"], columns=list("ABCD"))

def jaccard(M, i, j):
    a, b = M[i].notna(), M[j].notna()
    return (a & b).sum() / (a | b).sum()

def cosine_corated(M, i, j, center=False):
    X = M.sub(M.mean(axis=1), axis=0) if center else M      # center by USER mean -> adjusted cosine
    both = M[i].notna() & M[j].notna()
    a, b = X.loc[both, i], X.loc[both, j]
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Jaccard(A,D)         = {jaccard(M,'A','D'):.3f}")
print(f"Cosine(A,D) co-rated = {cosine_corated(M,'A','D'):.3f}")
print(f"Adjusted cosine(A,D) = {cosine_corated(M,'A','D',center=True):.3f}")

Jaccard(A,D)         = 0.750
Cosine(A,D) co-rated = 0.507
Adjusted cosine(A,D) = -0.943


### گام ۲ — Rating prediction

برای پیش‌بینی $\hat r_{ui}$: روی آیتم‌هایی که $u$ قبلاً rate کرده میانگین وزن‌دار بگیر؛ وزن = شباهت آن آیتم با $i$.

$$\hat r_{ui} = \frac{\sum_{j \in N_k(i;u)} \text{sim}(i,j)\cdot r_{uj}}{\sum_{j \in N_k(i;u)} |\text{sim}(i,j)|}$$

$N_k(i;u)$ = $k$ آیتمِ شبیه‌تر به $i$ از بین آیتم‌هایی که $u$ rate کرده.

```
PREDICT(u, i, k):
    for each item j rated by u:
        s[j] <- sim(i, j)
    N <- the k items with the largest s[j]          # professor's code: uses ALL rated items (no k)
    if sum(s[j] for j in N) == 0: return fallback   # professor returns 0; better: item/global mean
    return sum(s[j] * r[u,j] for j in N) / sum(|s[j]| for j in N)
```

**مثال دستی (Task 3 استاد):** $\hat r_{U2,B}$ با $k=2$، $J(B,A)=0.75$، $J(B,D)=0.5$، $r_{U2,A}=4$، $r_{U2,D}=2$:

$$\hat r = \frac{0.75\times4 + 0.5\times2}{0.75+0.5} = \frac{4.0}{1.25} = 3.2$$

**نسخه‌ی mean-centered** (در عمل بهتر، چون اختلاف سطح آیتم‌ها را حذف می‌کند):
$$\hat r_{ui} = \bar r_i + \frac{\sum_{j} \text{sim}(i,j)\,(r_{uj}-\bar r_j)}{\sum_{j} |\text{sim}(i,j)|}$$

**نقش $k$:** $k$ کوچک → نویزی (variance بالا)؛ $k$ خیلی بزرگ → همسایه‌های بی‌ربط وارد می‌شوند و جواب به میانگین کاربر میل می‌کند (bias بالا). معمولاً 20 تا 100، با validation انتخاب می‌شود.

### گام ۳ — Top-N، دو الگوریتم

**الگوریتم ۱: Predict & Sort** (ساده، کند)
```
TOPN_PREDICT_SORT(u, N):
    for each item i NOT rated by u:      # |I| items
        score[i] <- PREDICT(u, i)
    return the N items with the highest score
```

**الگوریتم ۲: kNN-map** (چیزی که واقعاً استفاده می‌شود)
```
OFFLINE:  knn_map[i] <- the k most similar items to i, with similarities     (for every item i)

TOPN_KNN_MAP(u, N):
    C <- empty map  item -> score
    for each item j rated by u:
        for each (sim, i) in knn_map[j]:
            if i not rated by u:
                C[i] += sim            # "highest total similarity"; alternatives: count, max
    return the N items in C with the highest score
```

**مثال دستی:** کاربر X و Y را دیده. `knn_map[X] = [(0.9, P), (0.5, Q)]`، `knn_map[Y] = [(0.6, Q), (0.4, R)]`.
جمع شباهت: P = 0.9، Q = 0.5+0.6 = **1.1**، R = 0.4 → ترتیب: **Q, P, R**.
(با معیار max: P=0.9, Q=0.6, R=0.4 → P, Q, R. با معیار count: Q=2, P=1, R=1.) این‌که سه معیار ترتیب متفاوت می‌دهند سوال امتحانی خوبی است.

### Complexity — جدول مهم

$n$ = تعداد آیتم، $m$ = تعداد کاربر، $\bar u$ = میانگین تعداد rating هر آیتم، $|I_u|$ = تعداد آیتم‌های کاربر.

| عملیات | هزینه | توضیح |
|---|---|---|
| `sim(i,j)` | $O(\bar u)$ | اشتراک دو لیست کاربر |
| `PREDICT(u,i)` | $O(\lvert I_u\rvert \cdot \bar u)$ | یک similarity برای هر آیتم کاربر |
| Top-N predict & sort | $O(n \cdot \lvert I_u\rvert \cdot \bar u)$ | برای **هر** درخواست! |
| ساخت kNN-map (brute force) | $O(n^2 \cdot \bar u)$ زمان، $O(n\,k)$ حافظه | یک بار، offline |
| Top-N با kNN-map | $O(\lvert I_u\rvert \cdot k)$ | مستقل از $n$ → online سریع |
| Amazon algorithm | بدترین حالت $O(n^2 m)$، در عمل $\approx O(n\,m)$ | فقط جفت‌های co-rated |

### Amazon item-to-item algorithm (Linden et al., 2003)

brute force همه‌ی $n^2$ جفت را بررسی می‌کند، ولی در داده‌ی sparse **بیشتر جفت‌ها هیچ کاربر مشترکی ندارند** (شباهت = 0). ایده: فقط جفت‌هایی را بشمار که واقعاً با هم دیده شده‌اند.

```
AMAZON_ITEM_TO_ITEM:
    for each item i:
        for each user u who rated i:
            for each item j rated by u (j != i):
                record that i and j were co-rated      # e.g. count[i][j] += 1
        for each item j recorded for i:
            compute sim(i, j)
        keep the top-k
```

**چرا «در عمل $O(nm)$»:** بیشتر کاربران تعداد کمی آیتم دارند، پس حلقه‌ی درونی کوتاه است. کاربران خیلی پرکار (که هزینه را بالا می‌برند) را معمولاً sample یا حذف می‌کنند.

سه راه دیگر برای فرار از $n^2$ که استاد اسم برده: **LSH / ANN** (تقریبی)، **ضرب ماتریسی روی GPU**، و **sampling** (کاری که کد استاد با `max_candidates=500` می‌کند).

## 2.C Professor's code — خط‌به‌خط

### ساختار داده‌ها

استاد دو dictionary می‌سازد تا lookup از $O(|R|)$ (فیلتر DataFrame) به $O(1)$ برسد:

- `rating_map["userId_movieId"] -> rating` — مثلاً `rating_map["1_101"] = 5.0`
- `user_ratings_map[movieId] -> (userids, ratings)` — همان «بردار آیتم» به شکل sparse

استاد این‌ها را با حلقه + `.iloc` / `.query` می‌سازد (کند: برای هر آیتم کل DataFrame اسکن می‌شود → $O(n\cdot|R|)$). همان خروجی با یک pass:

In [11]:
def build_maps(df):
    r_map  = {f"{u}_{i}": r for u, i, r in zip(df.userId, df.movieId, df.rating)}
    ur_map = {i: (g.userId.to_numpy(), g.rating.to_numpy()) for i, g in df.groupby("movieId")}
    return r_map, ur_map

rating_map, user_ratings_map = build_maps(ratings)
print(rating_map["1_101"], "|", user_ratings_map[10][0][:5], user_ratings_map[10][1][:5])

5.0 | [ 6  8 11 19 21] [3. 2. 3. 2. 5.]


### `calc_sim` و `NNCF_based_rating_prediction` (کد استاد، بدون تغییر در منطق)

In [12]:
def calc_sim(i, j, metric, ur_map):
    users_rated_i, ratings_i = ur_map[i]
    users_rated_j, ratings_j = ur_map[j]
    if metric == "Jaccard":                                   # ignores the rating VALUES
        inter = len(set(users_rated_i).intersection(users_rated_j))
        union = len(set(users_rated_i).union(users_rated_j))
        return 0 if union == 0 else inter / union
    elif metric == "Cosine":
        # common users + their positions in each array
        inter, ind1, ind2 = np.intersect1d(users_rated_i, users_rated_j, return_indices=True)
        if len(inter) == 0:
            return 0
        norm_i = np.linalg.norm(ratings_i[ind1])              # norms over CO-RATED users only  <-- key detail
        norm_j = np.linalg.norm(ratings_j[ind2])
        if norm_i == 0 or norm_j == 0:
            return 0
        return np.dot(ratings_i[ind1], ratings_j[ind2]) / (norm_i * norm_j)

def NNCF_based_rating_prediction(u, i, metric, ratings_df, r_map, ur_map):
    r, sum_sim = 0, 0
    movies_u = ratings_df[ratings_df["userId"] == u].movieId  # all items rated by u (no top-k!)
    for j in movies_u:
        sim = calc_sim(i, j, metric, ur_map)
        r += sim * r_map[f"{u}_{j}"]                           # numerator: sim * rating
        sum_sim += sim                                         # denominator: sum of sims
    return 0 if sum_sim == 0 else r / sum_sim

`np.intersect1d(..., return_indices=True)` سه چیز برمی‌گرداند: کاربران مشترک، و **اندیس** آن‌ها در آرایه‌ی اول و دوم — برای این‌که rating های دو آیتم را روی همان کاربران هم‌تراز کنیم. مثال خود استاد:

In [13]:
users_i, ratings_i = np.array([1, 3, 6, 9, 12, 15]), np.array([4, 3, 2, 4, 2, 5])
users_j, ratings_j = np.array([1, 6, 8, 12]),        np.array([2, 4, 2, 5])
inter, ind1, ind2 = np.intersect1d(users_i, users_j, return_indices=True)
print("common users:", inter, "| idx in i:", ind1, "| idx in j:", ind2)
print("aligned ratings:", ratings_i[ind1], ratings_j[ind2])
print("cosine =", 26, "/ (sqrt(24)*sqrt(45)) =", round(26 / (np.sqrt(24) * np.sqrt(45)), 3))

common users: [ 1  6 12] | idx in i: [0 2 4] | idx in j: [0 1 3]
aligned ratings: [4 2 2] [2 4 5]
cosine = 26 / (sqrt(24)*sqrt(45)) = 0.791


### سوال استاد: «How can you explain the difference between using Cosine vs. Jaccard?»

در خروجی استاد Jaccard (MAE ≈ 0.74) از Cosine (≈ 0.79) **بهتر** شد، با این‌که Jaccard اصلاً به مقدار rating نگاه نمی‌کند. اول آزمایش را تکرار کنیم (با test بزرگ‌تر، چون ۱۰۰ نمونه‌ی استاد خیلی نویزی است) و هم‌زمان توزیع similarity ها را هم نگه داریم:

In [14]:
Xtr_p, Xte_p = train_test_split(ratings, test_size=300, random_state=SEED)
train_r_map, train_ur_map = build_maps(Xtr_p)                 # maps from TRAIN only -> no leakage
mu_p = Xtr_p.rating.mean()
user_mean_p = Xtr_p.groupby("userId").rating.mean()

errs, sims = defaultdict(list), defaultdict(list)
for u, i, r in zip(Xte_p.userId, Xte_p.movieId, Xte_p.rating):
    for metric in ["Cosine", "Jaccard"]:
        try:    p = NNCF_based_rating_prediction(u, i, metric, Xtr_p, train_r_map, train_ur_map)
        except KeyError: p = mu_p                             # item never seen in train -> fallback
        errs[metric].append(abs(r - p))
    errs["Global average"].append(abs(r - mu_p))
    errs["User average"].append(abs(r - user_mean_p.get(u, mu_p)))
    if i in train_ur_map:                                     # keep the similarity distributions
        for j in Xtr_p[Xtr_p.userId == u].movieId.head(40):
            for metric in ["Cosine", "Jaccard"]:
                sims[metric].append(calc_sim(i, j, metric, train_ur_map))

print(pd.Series({k: np.mean(v) for k, v in errs.items()}, name="MAE").round(4).to_string())
sim_stats = pd.DataFrame({m: pd.Series(v)[pd.Series(v) > 0].describe()[["mean", "std", "25%", "50%", "75%"]] for m, v in sims.items()})
print("\nDistribution of the NON-ZERO similarities:"); print(sim_stats.round(3))
print("\nshare of non-zero cosine sims that are > 0.9 :", round(float((pd.Series(sims['Cosine'])[pd.Series(sims['Cosine'])>0] > 0.9).mean()), 3))

Cosine            0.7579
Jaccard           0.7203
Global average    0.8184
User average      0.7554

Distribution of the NON-ZERO similarities:
      Cosine  Jaccard
mean   0.961    0.124
std    0.038    0.092
25%    0.948    0.054
50%    0.968    0.104
75%    0.986    0.171

share of non-zero cosine sims that are > 0.9 : 0.938


**جواب (این را برای امتحان بلد باش):**

1. **Cosine استاد فقط روی کاربران مشترک حساب می‌شود و rating ها همه مثبت‌اند.** پس تقریباً همه‌ی شباهت‌ها نزدیک 1 درمی‌آیند (جدول بالا: میانگین بالا، پراکندگی خیلی کم). وقتی همه‌ی وزن‌ها تقریباً برابرند، میانگین وزن‌دار عملاً می‌شود **میانگین ساده‌ی rating های کاربر**؛ برای همین MAE کسینوس تقریباً با «User average» یکی است. یعنی cosine هیچ اطلاعاتی درباره‌ی این‌که «کدام آیتم‌ها به $i$ مربوط‌اند» اضافه نکرده.
2. **تعداد کاربران مشترک را نادیده می‌گیرد.** دو آیتم با یک کاربر مشترک cosine = 1.0 می‌گیرند — بیشترین وزن ممکن، از کم‌اعتمادترین شواهد.
3. **Jaccard** مقدار rating را دور می‌ریزد، ولی **قدرت co-occurrence** را اندازه می‌گیرد: آیتم‌هایی که توسط همان آدم‌ها دیده شده‌اند وزن بالا و بقیه وزن نزدیک صفر می‌گیرند. توزیعش پهن است، پس واقعاً بین همسایه‌ها تبعیض قائل می‌شود. «چه کسانی این را دیده‌اند» خودش سیگنال قوی سلیقه است.
4. **درمان cosine:** mean-centering (adjusted cosine / Pearson)، حساب کردن norm روی کل بردار (missing = 0)، یا **significance weighting / shrinkage**: $\text{sim}' = \frac{n_{ij}}{n_{ij}+\beta}\,\text{sim}$ که $n_{ij}$ تعداد کاربران مشترک است.

پایین در بخش 2.D می‌بینی که با این اصلاح‌ها cosine از Jaccard جلو می‌زند.

### Top-N — Predict & Sort (کد استاد)

```python
def top_N_pred_sort(N, u):
    preds = pd.Series([], dtype='float')
    all_items = set(ratings['movieId'].unique())
    rated_by_user = set(ratings[ratings["userId"]==1].movieId.unique())   # <-- BUG: hard-coded user 1, should be u
    not_rated_by_user = all_items - rated_by_user
    sample_movies = np.random.choice(list(not_rated_by_user), 500)        # only 500 sampled items, because it is too slow
    for m in tqdm(sample_movies):
        preds[m] = NNCF_based_rating_prediction(u, m, "Jaccard", ratings, rating_map, user_ratings_map)
    return preds.sort_values(ascending=False)[:N]
```

- **Bug:** `userId==1` به‌جای `u`؛ برای هر کاربری غیر از ۱، آیتم‌های «دیده‌نشده» اشتباه حساب می‌شود.
- خود استاد مجبور شده فقط ۵۰۰ آیتم تصادفی را امتیاز بدهد — یعنی عملاً نشان می‌دهد این الگوریتم scale نمی‌شود.

**سوال استاد در «Efficiency Issues»: «Can you suggest a solution?»** (برای این‌که rating همه‌ی آیتم‌ها را پیش‌بینی نکنیم)

> Do not score the whole catalog. Use a two-stage design: (1) **candidate generation** — cheaply select a few hundred candidates, e.g. the precomputed k nearest neighbours of the items the user already interacted with (kNN-map), or approximate nearest-neighbour search (LSH/HNSW) in an embedding space; (2) score/rank only those candidates. In addition, **precompute item–item similarities offline** and store only the top-k per item, so that no similarity is computed at request time.

### ساخت kNN-map (کد استاد)

```python
def build_knn_map(movies, K=30, max_candidates=500):
    knn_map = {}
    movie_ids = ratings['movieId'].unique()
    for i in tqdm(movie_ids):
        sorted_list = []                      # bounded sorted list of (sim, movieId), ascending
        knn_map[i] = sorted_list
        other_movies = movie_ids[movie_ids != i]
        candidate_list = np.random.choice(other_movies, max_candidates, replace=False)   # sampling for speed
        for j in candidate_list:
            sim = calc_sim(i, j, "Jaccard", user_ratings_map)
            if len(sorted_list) < K:
                bisect.insort(sorted_list, (sim, j))           # keep sorted
            elif sim > sorted_list[0][0]:                      # better than the current worst?
                bisect.insort(sorted_list, (sim, j))
                sorted_list.pop(0)                             # drop the smallest
    return knn_map
```

**چه کار می‌کند:** برای هر آیتم یک لیست مرتب به طول حداکثر $K$ نگه می‌دارد. `sorted_list[0]` همیشه **کم‌شباهت‌ترین** عضو فعلی است؛ اگر کاندید جدید از آن بهتر بود، وارد می‌شود و کوچک‌ترین حذف می‌شود. این الگوی کلاسیک **«top-K با حافظه‌ی $O(K)$»** است.

**نکته‌ها:**

- `max_candidates=500` یعنی هر آیتم فقط با ۵۰۰ آیتم **تصادفی** مقایسه می‌شود، نه همه → همسایه‌ها تقریبی‌اند و همسایه‌ی واقعی اغلب اصلاً دیده نمی‌شود. هزینه از $O(n^2)$ به $O(500\,n)$ می‌رسد.
- `bisect.insort` جای درج را با binary search در $O(\log K)$ پیدا می‌کند ولی خود درج در list و `pop(0)` هر دو $O(K)$ اند. ساختار درست‌تر **min-heap** است (`heapq.heappushpop` → $O(\log K)$). کل هزینه‌ی انتخاب top-K از $c$ کاندید: $O(c \log K)$ در برابر $O(c \log c)$ برای sort کامل.

نسخه‌ی heap برای یک آیتم، با **همه‌ی** کاندیدها (بدون sampling):

In [15]:
def knn_for_item(i, ur_map, K=10, metric="Jaccard"):
    heap = []                                                 # min-heap of (sim, item): heap[0] is the worst kept
    for j in ur_map:
        if j == i: continue
        s = calc_sim(i, j, metric, ur_map)
        if len(heap) < K:    heapq.heappush(heap, (s, j))
        elif s > heap[0][0]: heapq.heappushpop(heap, (s, j))  # push new, pop smallest: O(log K)
    return sorted(heap, reverse=True)

t0 = time.time()
nbrs = knn_for_item(1, user_ratings_map)                      # movieId 1 = Toy Story
print(f"one item vs all {n_items} items: {time.time()-t0:.2f}s  ->  full map ~ {(time.time()-t0)*n_items/60:.0f} min in pure Python\n")
for s, j in nbrs[:6]: print(f"  {s:.3f}  {title[j]}")

one item vs all 9724 items: 0.26s  ->  full map ~ 42 min in pure Python

  0.414  Independence Day (a.k.a. ID4) (1996)
  0.411  Jurassic Park (1993)
  0.404  Star Wars: Episode IV - A New Hope (1977)
  0.395  Forrest Gump (1994)
  0.393  Star Wars: Episode VI - Return of the Jedi (1983)
  0.391  Mission: Impossible (1996)


همسایه‌ها منطقی‌اند، ولی زمان را ببین: brute force در Python خالص برای کل map چند ده دقیقه طول می‌کشد. همین دلیلِ `max_candidates` استاد و دلیلِ بخش 2.D است.

### Amazon algorithm (کد استاد)

```python
users  = ratings.query("movieId == @i").userId.unique()        # users who rated i
movies = ratings.query("userId in @users").movieId.unique()    # every item those users rated = candidates
movies = np.random.choice(movies, 500)                         # speed-up, NOT part of the Amazon algorithm
for j in movies: ...same bounded sorted list...
```

تنها تفاوت با `build_knn_map`: کاندیدها به‌جای «۵۰۰ آیتم تصادفی از کل catalog»، از **آیتم‌هایی که حداقل یک کاربر مشترک با $i$ دارند** می‌آیند؛ هر چیز دیگری شباهتش صفر است و بررسی‌اش وقت تلف کردن است. استاد خودش می‌گوید روی MovieLens کمک زیادی نمی‌کند (چون نسبتاً dense است و آیتم‌های محبوب با تقریباً همه co-rated اند) ولی روی داده‌ی خیلی sparse مثل Amazon تفاوت عظیم است.

(دو نکته‌ی ریز: `np.random.choice(movies, 500)` با replacement است پس کاندید تکراری می‌دهد و می‌تواند یک همسایه را دو بار در لیست بگذارد؛ و متغیر `movies` ورودی تابع را overwrite می‌کند.)

پیاده‌سازی تمیز هسته‌ی الگوریتم (شمارش co-occurrence فقط برای جفت‌های واقعی):

In [16]:
def amazon_cooccurrence(item_users, user_items):
    """Linden et al. 2003: iterate item -> its users -> their items. Only co-rated pairs are ever touched."""
    co = {}
    for i, users in item_users.items():
        c = defaultdict(int)
        for u in users:
            for j in user_items[u]:
                if j != i: c[j] += 1
        co[i] = c
    return co

toy = M.notna()                                               # the 4x4 example matrix
item_users = {i: list(toy.index[toy[i]]) for i in toy.columns}
user_items = {u: list(toy.columns[toy.loc[u]]) for u in toy.index}
co = amazon_cooccurrence(item_users, user_items)
print("co-rating counts for A:", dict(co["A"]))
n_ = toy.sum()
print("Jaccard(A,D) from counts = co/(n_A+n_D-co) =", co["A"]["D"] / (n_["A"] + n_["D"] - co["A"]["D"]))

co-rating counts for A: {'B': 3, 'D': 3, 'C': 2}
Jaccard(A,D) from counts = co/(n_A+n_D-co) = 0.75


### Top-N با kNN-map (کد استاد) — و یک bug مهم

```python
def add_sims_and_sort(l):
    li = []
    it = itertools.groupby(l, operator.itemgetter(1))        # group by movieId
    for key, subiter in it:
        li.append((key, sum(item[0] for item in subiter)))   # sum the similarities per movie
    return sorted(li, key=itemgetter(1), reverse=True)

def top_N_knn_map(ratings, N, u):
    C = []
    for m in ratings.query("userId == @u").movieId:
        C = C + knn_map[m]                                    # concatenate neighbour lists
    return add_sims_and_sort(C)[:N]
```

هدف: «highest total similarity» — جمع شباهت هر کاندید روی همه‌ی آیتم‌های کاربر.

**Bug:** `itertools.groupby` فقط عناصر **پشت‌سرهمِ** هم‌کلید را گروه می‌کند، پس ورودی باید از قبل بر اساس کلید sort شده باشد. اینجا `C` بر اساس movieId مرتب نیست، در نتیجه شباهت‌های یک فیلم که از چند همسایه آمده **با هم جمع نمی‌شوند** و فیلم چند بار جدا در لیست می‌ماند. عملاً کد استاد «max similarity» را حساب می‌کند، نه «total similarity». نشانه‌اش در خروجی خود استاد: همه‌ی score ها زیر 1 اند.

In [17]:
C = [(0.9, "P"), (0.5, "Q"), (0.4, "R"), (0.6, "Q")]      # concatenated neighbour lists; the two Q entries are NOT adjacent

def add_sims_and_sort_prof(l):
    li = [(key, sum(x[0] for x in sub)) for key, sub in itertools.groupby(l, operator.itemgetter(1))]
    return sorted(li, key=operator.itemgetter(1), reverse=True)

def add_sims_and_sort_fixed(l):
    acc = defaultdict(float)
    for s, item in l: acc[item] += s                          # dict accumulation: order does not matter
    return sorted(acc.items(), key=operator.itemgetter(1), reverse=True)

print("professor:", add_sims_and_sort_prof(C))
print("fixed    :", add_sims_and_sort_fixed(C))

professor: [('P', 0.9), ('Q', 0.6), ('Q', 0.5), ('R', 0.4)]
fixed    : [('Q', 1.1), ('P', 0.9), ('R', 0.4)]


**سه ایراد دیگر همین تابع** (هر کدام می‌تواند سوال «what is wrong with this code» باشد):

1. **فیلم‌های دیده‌شده حذف نمی‌شوند** — ممکن است چیزی را که کاربر قبلاً دیده پیشنهاد بدهد.
2. **Leakage در evaluation:** `knn_map` روی **کل** `ratings` ساخته شده، بعد test جدا می‌شود؛ یعنی شباهت‌ها rating های test را دیده‌اند. باید knn-map فقط از train ساخته شود.
3. کامنت می‌گوید `ratings >= 4` ولی کد `rating > 4` است (یعنی فقط 4.5 و 5).

### Hit Ratio (کد استاد)

این‌جا پروتکل با Part 1 فرق دارد: **بدون negative sampling**. برای هر $(u,i)$ در test یک لیست top-N واقعی ساخته می‌شود و چک می‌شود $i$ در آن هست یا نه.
$$\text{Hit Ratio} = \frac{\#\text{hits}}{|T|}$$
استاد با $N=100$ عدد 0.165 گرفت. توجه کن که این عدد با HR های Part 1 **قابل مقایسه نیست** (آن‌جا رتبه بین ۱۰۰ کاندید بود، این‌جا بین کل catalog).

## 2.D Industry version

ایده‌ی اصلی: **کل الگوریتم item-kNN چند ضرب sparse matrix است.** اگر $X$ ماتریس user×item باشد و ستون‌هایش L2-normalize شده باشند:

$$S = X^\top X \quad\Rightarrow\quad S_{ij} = \cos(i,j) \qquad\qquad \text{scores} = X \cdot S_k^\top$$

و ضرب sparse خودبه‌خود فقط جفت‌های co-rated را لمس می‌کند — یعنی **همان Amazon algorithm، ولی در C**.

### گام ۱ — ماتریس sparse

In [18]:
user_ids, item_ids = np.sort(ratings.userId.unique()), np.sort(ratings.movieId.unique())
u2idx = pd.Series(np.arange(n_users), index=user_ids)        # raw id -> contiguous index (always needed in practice)
i2idx = pd.Series(np.arange(n_items), index=item_ids)

def to_csr(df, binary=False):
    v = np.ones(len(df)) if binary else df.rating.to_numpy(float)
    return csr_matrix((v, (u2idx[df.userId].to_numpy(), i2idx[df.movieId].to_numpy())), shape=(n_users, n_items))

X_all = to_csr(ratings)
print(repr(X_all))
print(f"memory: sparse {X_all.data.nbytes/1e6:.1f} MB  vs  dense {n_users*n_items*8/1e6:.1f} MB")

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 100836 stored elements and shape (610, 9724)>
memory: sparse 0.8 MB  vs  dense 47.5 MB


### گام ۲ — top-k similarity برای همه‌ی آیتم‌ها (chunked)

ماتریس کامل $n \times n$ را نگه نمی‌داریم؛ chunk به chunk حساب می‌کنیم و فقط top-k هر سطر را برمی‌داریم (حافظه $O(nk)$).

In [19]:
def topk_item_sim(X, k=30, metric="cosine", center=None, shrink=0, chunk=1000):
    """X: users x items (CSR). Returns sparse items x items matrix, row i = top-k neighbours of item i.
    metric : 'cosine' (missing = 0, full-vector norms) or 'jaccard'
    center : None | 'user'  -> subtract the user's mean from each rating first (adjusted cosine)
    shrink : beta in  sim * n_ij / (n_ij + beta)   (significance weighting)"""
    X = X.tocsr().astype(float)
    B = X.copy(); B.data[:] = 1.0                             # binary "who rated what"
    if center == "user":
        X = X.copy()
        means = np.asarray(X.sum(1)).ravel() / np.maximum(np.diff(X.indptr), 1)
        X.data -= np.repeat(means, np.diff(X.indptr))
    Bi, cnt = B.T.tocsr(), np.asarray(B.sum(0)).ravel()       # items x users, #ratings per item
    Xi = Bi if metric == "jaccard" else normalize(X.T.tocsr())
    rows, cols, vals = [], [], []
    for s in range(0, X.shape[1], chunk):
        e   = min(s + chunk, X.shape[1])
        co  = (Bi[s:e] @ Bi.T).toarray()                      # co-rating counts n_ij  (sparse product = Amazon trick)
        sim = co / np.maximum(cnt[s:e, None] + cnt[None, :] - co, 1) if metric == "jaccard" else (Xi[s:e] @ Xi.T).toarray()
        if shrink: sim = sim * co / (co + shrink)
        sim[np.arange(e - s), np.arange(s, e)] = 0            # no self-similarity
        idx = np.argpartition(-sim, k, axis=1)[:, :k]         # top-k per row in O(n), no full sort
        rows.append(np.repeat(np.arange(s, e), k)); cols.append(idx.ravel()); vals.append(np.take_along_axis(sim, idx, 1).ravel())
    S = csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(X.shape[1],) * 2)
    S.data[S.data < 0] = 0; S.eliminate_zeros()               # keep positive neighbours only
    return S

t0 = time.time()
S_demo = topk_item_sim(to_csr(ratings, binary=True), k=10, metric="jaccard")
print(f"exact Jaccard top-10 for ALL {n_items} items: {time.time()-t0:.1f}s   (pure-Python estimate above: tens of minutes)\n")
row = S_demo[i2idx[1]]
for j, s in sorted(zip(row.indices, row.data), key=lambda t: -t[1])[:6]: print(f"  {s:.3f}  {title[item_ids[j]]}")

exact Jaccard top-10 for ALL 9724 items: 2.5s   (pure-Python estimate above: tens of minutes)

  0.414  Independence Day (a.k.a. ID4) (1996)
  0.411  Jurassic Park (1993)
  0.404  Star Wars: Episode IV - A New Hope (1977)
  0.395  Forrest Gump (1994)
  0.393  Star Wars: Episode VI - Return of the Jedi (1983)
  0.391  Mission: Impossible (1996)


همان همسایه‌های نسخه‌ی Python خالص، ولی برای **کل catalog** در چند ثانیه و **دقیق** (بدون sampling).

### گام ۳ — Rating prediction و مقایسه‌ی similarity ها

$$\text{num} = X\,S_k^\top,\qquad \text{den} = B\,S_k^\top,\qquad \hat R = \text{num} / \text{den}$$

In [20]:
def knn_predict(Xtr, S, tu, ti, fallback, item_mean=None):
    """r_hat(u,i) = sum_j S[i,j] r_uj / sum_j S[i,j]  over j in (top-k of i) ∩ (rated by u). Optional item-mean centering."""
    B = Xtr.copy(); B.data[:] = 1.0
    V = Xtr
    if item_mean is not None:
        V = Xtr.copy(); V.data -= item_mean[V.indices]        # r_uj - mean_j
    num = np.asarray((V @ S.T)[tu, ti]).ravel()
    den = np.asarray((B @ S.T)[tu, ti]).ravel()
    ok  = den > 1e-9
    pred = fallback.copy()
    pred[ok] = num[ok] / den[ok] + (item_mean[ti][ok] if item_mean is not None else 0)
    return np.clip(pred, 0.5, 5), ok.mean()

Xtr = to_csr(X_train)                                         # same 90/10 split as Part 1
tu, ti, y = u2idx[X_test.userId].to_numpy(), i2idx[X_test.movieId].to_numpy(), X_test.rating.to_numpy()
cnt_i  = np.diff(Xtr.tocsc().indptr)
imean  = np.where(cnt_i > 0, np.asarray(Xtr.sum(0)).ravel() / np.maximum(cnt_i, 1), mu)
fb     = preds["Bias: mu+b_u+b_i"].astype(float)              # fallback when no neighbour is available

K_PRED = 40
variants = {
    "Jaccard":                                dict(metric="jaccard"),
    "Cosine (missing=0)":                     dict(metric="cosine"),
    "Adjusted cosine + shrink":               dict(metric="cosine", center="user", shrink=20),
}
rows = {}
for name, kw in variants.items():
    S = topk_item_sim(Xtr, k=K_PRED, **kw)
    p, cov = knn_predict(Xtr, S, tu, ti, fb)
    rows[f"item-kNN | {name}"] = {"MAE": mae(y, p), "RMSE": rmse(y, p), "coverage": cov}
p, cov = knn_predict(Xtr, S, tu, ti, fb, item_mean=imean)     # last S = adjusted cosine
rows["item-kNN | Adjusted cosine + shrink + mean-centered prediction"] = {"MAE": mae(y, p), "RMSE": rmse(y, p), "coverage": cov}
knn_table = pd.concat([rating_table.assign(coverage=1.0).loc[["Global mean", "User mean", "Bias: mu+b_u+b_i"]], pd.DataFrame(rows).T])
knn_table

,MAE,RMSE,coverage
Global mean,0.8351,1.0506,1.0000
User mean,0.7432,0.9563,1.0000
Bias: mu+b_u+b_i,0.6796,0.8861,1.0000
item-kNN | Jaccard,0.6767,0.9036,0.8460
item-kNN | Cosine (missing=0),0.6775,0.9079,0.8148
item-kNN | Adjusted cosine + shrink,0.6618,0.8930,0.9111
item-kNN | Adjusted cosine + shrink + mean-centered prediction,0.6602,0.8795,0.9111


`coverage` = سهمی از test که حداقل یک همسایه داشت (بقیه به bias baseline برمی‌گردند).

**چیزهایی که باید از جدول بگیری:**

- هر چه similarity «هوشمندتر» می‌شود (mean-centering + shrinkage) و هر چه prediction هم mean-centered می‌شود، خطا پایین‌تر می‌آید. این همان مسیری است که بخش 2.B گفت.
- **Bias baseline رقیب سرسختی است.** kNN در MAE کمی بهتر است ولی در RMSE فقط بهترین نسخه به آن می‌رسد؛ بهترین نسخه‌های kNN (مثل `KNNBaseline` پایین) در واقع kNN را **روی باقی‌مانده‌ی baseline** سوار می‌کنند.

### Surprise — همان الگوریتم در سه خط

In [21]:
from surprise import KNNBasic, KNNWithMeans, KNNBaseline

algos = {
    "KNNBasic  (cosine)":            KNNBasic(k=40, sim_options={"name": "cosine", "user_based": False}, verbose=False),
    "KNNWithMeans (pearson)":        KNNWithMeans(k=40, sim_options={"name": "pearson", "user_based": False}, verbose=False),
    "KNNBaseline (pearson_baseline)": KNNBaseline(k=40, sim_options={"name": "pearson_baseline", "user_based": False, "shrinkage": 100}, verbose=False),
}
for name, algo in algos.items():
    cv = cross_validate(algo, sdata, measures=["MAE", "RMSE"], cv=3, verbose=False)
    print(f"{name:32s} MAE={cv['test_mae'].mean():.4f}  RMSE={cv['test_rmse'].mean():.4f}")

KNNBasic  (cosine)               MAE=0.7603  RMSE=0.9761


KNNWithMeans (pearson)           MAE=0.6948  RMSE=0.9131


KNNBaseline (pearson_baseline)   MAE=0.6594  RMSE=0.8643


`user_based: False` یعنی item-based. `KNNBasic` با cosine همان similarity استاد است (cosine خام روی co-rated ها) — و می‌بینی که بدترین است. `pearson_baseline` با `shrinkage` همان درمان‌هایی است که در جواب سوال استاد نوشتم.

### گام ۴ — Top-N با kNN-map، به شکل ماتریسی

$$\text{score}(u, i) = \sum_{j \in I_u} S_k[j, i] \quad\Longleftrightarrow\quad \text{Scores} = B \cdot S_k$$

این دقیقاً «highest total similarity» استاد است (نسخه‌ی درست، بدون bug). ارزیابی: leave-one-out از Part 1، ولی این بار **full ranking** روی کل catalog (بدون negative sampling) و knn-map **فقط از train**.

In [22]:
def hit_ratio_full(scores, B_train, test, Ks=(10, 50, 100)):
    """Full-catalog ranking. scores: users x items (dense). Seen items are masked out."""
    sc = np.array(scores, dtype=float)
    sc[B_train.nonzero()] = -np.inf                           # never recommend what the user already has
    tu, ti = u2idx[test.userId].to_numpy(), i2idx[test.movieId].to_numpy()
    rank = (sc[tu] > sc[tu, ti][:, None]).sum(1)              # rank of the held-out item for each test user
    return {f"HR@{k}": float((rank < k).mean()) for k in Ks}

B_tr = to_csr(tr, binary=True)                                # tr / te: leave-one-out split from Part 1
pop_scores = np.tile(np.asarray(B_tr.sum(0)).ravel(), (n_users, 1))

S_jac = topk_item_sim(B_tr, k=30, metric="jaccard")
S_cos = topk_item_sim(B_tr, k=30, metric="cosine")            # cosine on implicit 0/1 data
topn_full = {
    "Popularity":                 hit_ratio_full(pop_scores, B_tr, te),
    "item-kNN Jaccard (k=30)":    hit_ratio_full((B_tr @ S_jac).toarray(), B_tr, te),
    "item-kNN Cosine  (k=30)":    hit_ratio_full((B_tr @ S_cos).toarray(), B_tr, te),
}
pd.DataFrame(topn_full).T

,HR@10,HR@50,HR@100
Popularity,0.1571,0.3281,0.4206
item-kNN Jaccard (k=30),0.2356,0.4468,0.5515
item-kNN Cosine (k=30),0.2461,0.4538,0.5462


حالا kNN شخصی‌سازی‌شده popularity را در full ranking **می‌برد** — چیزی که از عدد 0.165 استاد (با sampling و bug) معلوم نبود.

نسخه‌ی dictionary همان kNN-map (شکل استاد) برای یک کاربر، با aggregation درست و حذف دیده‌شده‌ها:

In [23]:
knn_map = {item_ids[i]: [(s, item_ids[j]) for j, s in zip(S_cos[i].indices, S_cos[i].data)] for i in range(n_items)}

def top_N_knn_map(train_df, N, u, agg="sum"):
    seen, acc = set(train_df.loc[train_df.userId == u, "movieId"]), defaultdict(float)
    for m in seen:
        for s, j in knn_map.get(m, []):
            if j in seen: continue                            # filter already-watched items
            acc[j] = acc[j] + s if agg == "sum" else max(acc[j], s) if agg == "max" else acc[j] + 1
    return heapq.nlargest(N, acc.items(), key=operator.itemgetter(1))

U = 100
print("User", U, "liked:", "; ".join(title[m] for m in tr[(tr.userId == U) & (tr.rating >= 5)].movieId.head(5)))
for m, s in top_N_knn_map(tr, 8, U): print(f"  {s:6.2f}  {title[m]}")

User 100 liked: Top Gun (1986); Christmas Vacation (National Lampoon's Christmas Vacation) (1989); Officer and a Gentleman, An (1982); Sweet Home Alabama (2002)
   17.48  Indiana Jones and the Temple of Doom (1984)
   15.97  Big (1988)
   15.74  Ferris Bueller's Day Off (1986)
   12.12  Monty Python and the Holy Grail (1975)
   10.87  Who Framed Roger Rabbit? (1988)
   10.76  E.T. the Extra-Terrestrial (1982)
   10.63  Beetlejuice (1988)
   10.12  Star Wars: Episode VI - Return of the Jedi (1983)


### `implicit` — کتابخانه‌ای که واقعاً در production استفاده می‌شود

`implicit` (C++/Cython، چندنخی، GPU اختیاری) برای **implicit feedback و Top-N** ساخته شده. item-kNN اش همین الگوریتم است؛ نسخه‌های وزن‌دار هم دارد (`TfidfRecommender`، `BM25Recommender`) که باید روی داده‌ی خودت tune و مقایسه شوند.

In [24]:
from implicit.nearest_neighbours import CosineRecommender

def implicit_scores(model, B_train, N=100):
    """Turn implicit's top-N output into a users x items score matrix so that we can reuse hit_ratio_full."""
    ids, sc = model.recommend(np.arange(n_users), B_train, N=N, filter_already_liked_items=True)
    out = np.full((n_users, n_items), -1e9)
    np.put_along_axis(out, ids, sc, axis=1)
    return out

knn_model = CosineRecommender(K=30)
knn_model.fit(B_tr, show_progress=False)                      # user x item CSR
topn_full["implicit CosineRecommender (K=30)"] = hit_ratio_full(implicit_scores(knn_model, B_tr), B_tr, te)
pd.DataFrame(topn_full).T

,HR@10,HR@50,HR@100
Popularity,0.1571,0.3281,0.4206
item-kNN Jaccard (k=30),0.2356,0.4468,0.5515
item-kNN Cosine (k=30),0.2461,0.4538,0.5462
implicit CosineRecommender (K=30),0.2426,0.4538,0.5462


عدد `implicit` با پیاده‌سازی دستی ما تقریباً یکی است — یعنی همان الگوریتم است، فقط سریع‌تر و آماده‌ی production.

In [25]:
# "Because you watched ..." : آیتم‌های مشابه با یک خط
ids, sc = knn_model.similar_items(int(i2idx[1]), N=6)
print("Similar to:", title[1]); [print(f"  {s:.3f}  {title[item_ids[j]]}") for j, s in zip(ids[1:], sc[1:])];

Similar to: Toy Story (1995)
  0.578  Star Wars: Episode IV - A New Hope (1977)
  0.574  Independence Day (a.k.a. ID4) (1996)
  0.574  Forrest Gump (1994)
  0.574  Jurassic Park (1993)
  0.565  Star Wars: Episode VI - Return of the Jedi (1983)


### وقتی catalog میلیونی است: ANN با FAISS

همه‌ی کدهای بالا **exact** اند و $O(n^2)$. برای میلیون‌ها آیتم، بردار هر آیتم (معمولاً embedding فشرده، نه بردار خام کاربران) را در یک **Approximate Nearest Neighbor index** می‌ریزند: FAISS (Meta)، ScaNN (Google)، HNSW (`hnswlib`)، Annoy (Spotify). این همان خانواده‌ی «LSH / approximation» است که استاد اسم برد. جست‌وجو از $O(n)$ به حدود $O(\log n)$ می‌رسد، با کمی خطا.

In [26]:
import faiss
item_vecs = normalize(B_tr.T.tocsr()).toarray().astype("float32")    # demo: raw item vectors (610-d). Real systems: 64-256-d embeddings
index = faiss.IndexHNSWFlat(item_vecs.shape[1], 32, faiss.METRIC_INNER_PRODUCT)   # graph-based ANN index
index.add(item_vecs)
sims_, nbrs_ = index.search(item_vecs[[i2idx[1]]], 6)                # query = Toy Story
print("FAISS HNSW neighbours of:", title[1]); [print(f"  {s:.3f}  {title[item_ids[j]]}") for s, j in zip(sims_[0][1:], nbrs_[0][1:])];

FAISS HNSW neighbours of: Toy Story (1995)
  0.578  Star Wars: Episode IV - A New Hope (1977)
  0.574  Independence Day (a.k.a. ID4) (1996)
  0.574  Forrest Gump (1994)
  0.574  Jurassic Park (1993)
  0.565  Star Wars: Episode VI - Return of the Jedi (1983)


### جمع‌بندی صنعتی Part 2

| نیاز | ابزار |
|---|---|
| Rating prediction، آموزش و prototype | `scikit-surprise` |
| Top-N روی implicit data، production در scale متوسط | `implicit` |
| داده‌ی خیلی بزرگ (distributed) | Spark MLlib (ALS)، یا job های SQL/Spark برای co-occurrence |
| Nearest-neighbor search در scale | FAISS، ScaNN، HNSW |
| معماری | offline: ساخت similarity/index (batch) → online: lookup + ranking |

## 2.E Exam — kNN CF

### سوال‌های استاد (Exercises) با جواب

**Q1. How does Jaccard differ from Cosine in CF regarding rating values? Which one ignores the magnitude?**
Jaccard treats ratings as binary (rated / not rated): it only compares the *sets* of users who interacted with the two items, $|U_i\cap U_j|/|U_i\cup U_j|$, so it **ignores the rating magnitudes** (1 star and 5 stars count the same). Cosine treats each item as a vector of rating values and measures the angle between the vectors, so the actual values matter.
*Extra credit:* raw cosine on strictly positive ratings cannot express disagreement (all products are positive); mean-centering (adjusted cosine / Pearson) fixes this. Jaccard is a natural fit for implicit feedback.

**Q2. Time complexity of building a full kNN map for $N$ items? Why is this an inefficiency?**
All pairwise similarities must be computed: $N(N-1)/2$ pairs → $O(N^2)$ similarity computations (each costs $O(\bar u)$, so $O(N^2\bar u)$ overall). Quadratic growth makes it infeasible for millions of items (10⁶ items → ~5·10¹¹ pairs), and the map must be rebuilt as data changes. Remedies: approximate NN search (LSH), GPU matrix multiplication, the Amazon item-to-item algorithm that skips pairs with no common user, or candidate sampling. Storage is only $O(Nk)$ because just the top-$k$ neighbours are kept.

**Q3. Difference between MAE and Hit Ratio; when is each appropriate?**
MAE measures the average absolute difference between predicted and true ratings — a *regression* error for the **rating prediction** task (lower is better). Hit Ratio measures whether a held-out relevant item appears in the top-N list — a *ranking* metric for the **Top-N recommendation** task (higher is better). MAE needs explicit ratings and treats all items equally; Hit Ratio only cares about the top of the list, which is what the user actually sees, and also works with implicit feedback.

**Q4.** → حل کامل در بخش 2.B: Jaccard(A,D) = **0.75**؛ Cosine(A,D) = 18/√1260 = **0.507**؛ $\hat r_{U2,B}$ = 4.0/1.25 = **3.2**.

**Q5. Hit Ratio.** U1: hit (Movie 10 in list) · U2: miss · U3: hit · U4: hit · U5: miss → HR = 3/5 = **0.60**.

**In-text question — Cosine vs Jaccard result.** → جواب کامل در 2.C. نسخه‌ی کوتاه امتحانی:
> In the notebook, cosine is computed only over co-rated users and all ratings are positive, so almost every similarity is close to 1 (and exactly 1 when there is a single common user). The weights are nearly uniform, so the prediction degenerates to the user's average rating. Jaccard ignores rating values but captures how strongly two items co-occur; its values are spread out, so it actually discriminates between relevant and irrelevant neighbours. Hence Jaccard gives a lower MAE here. Mean-centering and significance weighting would fix cosine.

**In-text question — Efficiency: "Can you suggest a solution?"** → جواب در 2.C (candidate generation + precomputed top-k similarities + ANN).

### سوال‌های محتمل دیگر — الگوریتمی

**A1. Write the item-based rating prediction algorithm in pseudocode and give its complexity.** → `PREDICT` در 2.B؛ $O(|I_u|\cdot\bar u)$.

**A2. Write the Top-N algorithm using a kNN map. What are the offline and online parts?** → `TOPN_KNN_MAP` در 2.B. Offline: building the map, $O(n^2\bar u)$ time, $O(nk)$ space. Online: $O(|I_u|\cdot k)$ per request, independent of catalog size.

**A3. Compare "Predict & Sort" with the kNN-map approach.**
Predict & Sort predicts a rating for every unseen item ($O(n\,|I_u|\,\bar u)$ per request) and recomputes the same similarities for every user. The kNN-map approach precomputes similarities once, stores only the top-$k$ per item, and at request time only looks at neighbours of the user's items — orders of magnitude faster, at the cost of an offline build and approximate scores.

**A4. Explain the Amazon item-to-item algorithm and why it is faster on sparse data.** → pseudocode در 2.B.
It only computes similarities for item pairs that share at least one user, by iterating item → users who bought it → other items those users bought. In sparse data the vast majority of pairs have no common user, so they are skipped. Worst case $O(N^2M)$, in practice close to $O(NM)$ because most users have few interactions.

**A5. How do you keep the k most similar items efficiently while scanning candidates?**
Keep a bounded structure of size $k$ whose smallest element is cheap to access (sorted list as in the notebook, or a min-heap). For each candidate, if its similarity exceeds the current minimum, insert it and remove the minimum. Memory $O(k)$; with a heap the total time is $O(c\log k)$ for $c$ candidates instead of $O(c\log c)$ for a full sort.

**A6. Given `knn_map[X] = [(0.9,P),(0.5,Q)]`, `knn_map[Y] = [(0.6,Q),(0.4,R)]` and a user who watched X and Y, give the top-2 under (a) total similarity, (b) max similarity, (c) frequency.**
(a) Q=1.1, P=0.9 → [Q, P]. (b) P=0.9, Q=0.6 → [P, Q]. (c) Q=2, then P or R (tie) → [Q, P/R].

**A7. Compute the adjusted cosine between A and D in the Q4 matrix.** → 2.B: **−0.943**. Interpretation: the items are strongly *dis*similar although Jaccard (0.75) and raw cosine (0.507) both suggest similarity.

**A8. Using the Q4 matrix, predict User 1's rating for Item C with Jaccard and all rated items.**
$U_C=\{2,4\}$. $J(C,A)=|\{2,4\}|/|\{1,2,3,4\}|=0.5$; $J(C,B)=|\{4\}|/|\{1,2,3,4\}|=0.25$; $J(C,D)=|\{2\}|/|\{1,2,3,4\}|=0.25$.
$\hat r = \dfrac{0.5\cdot5+0.25\cdot3+0.25\cdot1}{0.5+0.25+0.25} = \dfrac{3.5}{1.0} = 3.5$.

### سوال‌های محتمل دیگر — مفهومی

**B1. Why is mean-centering needed for cosine on explicit ratings?**
Ratings are strictly positive, so every term of the dot product is positive and disagreement (5 vs 1) still increases similarity — it even contributes more than agreeing on a low rating (1 vs 1). Subtracting the user's mean makes "liked" positive and "disliked" negative, so agreement adds and disagreement subtracts. It also removes user rating-scale bias (strict vs generous raters).

**B2. When is raw cosine appropriate?** Implicit binary feedback: $1\times1$ counts a co-occurrence and everything else is 0; there is no notion of a negative rating.

**B3. User-based vs item-based CF — why did industry prefer item-based?**
There are usually fewer items than users and item–item similarities are more stable over time, so they can be precomputed offline and reused; a user's vector changes with every interaction. Item-based recommendations are also easy to explain.

**B4. What is the effect of $k$?** Small $k$: noisy, high variance, low coverage. Large $k$: includes weakly related neighbours, predictions regress toward the user's mean. Tuned on a validation set.

**B5. Why are `rating_map` and `user_ratings_map` built?** To replace repeated DataFrame scans with $O(1)$ dictionary lookups inside the nested similarity/prediction loops.

**B6. What are the limitations of neighbourhood CF?** Cold start for new users/items; sparsity (few co-ratings → unreliable similarities); $O(n^2)$ offline cost; popularity bias; no use of content or context.

**B7. Is the Hit Ratio of Part 1 (1+99 negatives) comparable to the Hit Ratio in the kNN notebook?** No. The first ranks the positive among 100 sampled candidates; the second requires it to appear in a top-N list drawn from the entire catalog. Only numbers obtained under the same protocol are comparable.

**B8 (code reading). What is wrong with `add_sims_and_sort`?** `itertools.groupby` only groups *consecutive* equal keys, so the list must be sorted by movieId first; otherwise similarities of the same movie are not summed. Fix: sort by key before grouping, or accumulate in a dictionary.

---
# Part 3 — Matrix Factorization, BPR and LightGCN

> این بخش از `gnn.ipynb` است. در آن نوت‌بوک تقریباً هیچ توضیحی درباره‌ی خود الگوریتم‌ها نیست (فقط کد + دو یادداشت ریاضی)، پس اینجا الگوریتم را کامل باز می‌کنم.

## 3.A Problem — چرا از kNN جلوتر برویم؟

kNN سه محدودیت دارد:

1. **Sparsity:** دو آیتم که هیچ کاربر مشترکی ندارند شباهت‌شان صفر است، حتی اگر واقعاً شبیه باشند.
2. **Scale:** $O(n^2)$ برای ساخت و $O(nk)$ حافظه.
3. **فقط یک قدم:** فقط co-occurrence مستقیم را می‌بیند.

**راه‌حل: model-based CF.** به‌جای نگه داشتن شباهت‌ها، برای هر user و هر item یک بردار کوتاه (**embedding**، مثلاً 64 بعدی) **یاد می‌گیریم** طوری که ضرب داخلی‌شان علاقه را پیش‌بینی کند. دو آیتم بدون کاربر مشترک هم می‌توانند embedding نزدیک بگیرند.

**تغییر دوم در این نوت‌بوک: implicit feedback.** دیتاست Gowalla check-in است، نه ستاره. فقط می‌دانیم «کاربر به این مکان رفته» (positive)؛ درباره‌ی بقیه **نمی‌دانیم** دوست نداشته یا فقط ندیده. پس:

- MAE/RMSE بی‌معنی است → متریک‌های ranking: **Precision@K، Recall@K، NDCG@K**
- loss رگرسیونی نداریم → **BPR**، یک loss مخصوص ranking
- نمونه‌ی منفی نداریم → **negative sampling**

**کِی از چه استفاده می‌شود:**

| روش | کاربرد واقعی |
|---|---|
| MF (ALS/BPR) | workhorse صنعت برای implicit data؛ candidate generation؛ embedding برای ANN |
| LightGCN / GNN | وقتی ساختار گراف مهم است و منابع train داریم (PinSage در Pinterest نمونه‌ی معروف GNN در production است) |

## 3.B Algorithms

### Metrics — با مثال دستی

برای یک کاربر: `predicted` = لیست رتبه‌بندی‌شده‌ی top-K، `actual` = مجموعه‌ی آیتم‌های test او.

$$\text{Precision@}K = \frac{|\text{top-}K \cap \text{actual}|}{K} \qquad \text{Recall@}K = \frac{|\text{top-}K \cap \text{actual}|}{|\text{actual}|}$$

$$\text{DCG@}K = \sum_{p=1}^{K} \frac{rel_p}{\log_2(p+1)} \qquad \text{IDCG@}K = \sum_{p=1}^{\min(K,|\text{actual}|)} \frac{1}{\log_2(p+1)} \qquad \text{NDCG@}K = \frac{\text{DCG}}{\text{IDCG}}$$

- Precision: چه کسری از پیشنهادها درست بود. Recall: چه کسری از آیتم‌های درست را پیدا کردیم.
- هر دو به **ترتیب داخل top-K** حساس نیستند. **NDCG** هست: hit در رتبه‌ی ۱ ارزش 1 دارد، رتبه‌ی ۲ ارزش $1/\log_2 3 = 0.63$، رتبه‌ی ۳ ارزش $0.5$ و …
- IDCG = DCG بهترین ترتیب ممکن؛ تقسیم بر آن NDCG را بین 0 و 1 می‌آورد.
- در کد استاد اندیس از 0 شروع می‌شود، برای همین `1/log2(i+2)` نوشته شده.

**مثال:** `predicted = [a, b, c, d, e]`، `actual = {b, e, z}`، $K=5$.

- hit ها: b (رتبه 2) و e (رتبه 5) → Precision@5 $= 2/5 = 0.4$، Recall@5 $= 2/3 = 0.667$
- DCG $= \frac{1}{\log_2 3} + \frac{1}{\log_2 6} = 0.631 + 0.387 = 1.018$
- IDCG (۳ آیتم relevant در ۳ رتبه‌ی اول) $= 1 + 0.631 + 0.5 = 2.131$
- NDCG@5 $= 1.018 / 2.131 = 0.478$

**رابطه با Hit Ratio:** اگر هر کاربر فقط یک آیتم test داشته باشد، Recall@K همان HR@K است.

In [27]:
# توابع استاد (بدون تغییر)
def recall_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    return len(set(predicted[:k]) & set(actual)) / len(set(actual))

def precision_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    return len(set(predicted[:k]) & set(actual)) / k

def ndcg_at_k(actual, predicted, k):
    if len(actual) == 0: return 0.0
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(actual), k)))
    dcg  = sum(1.0 / math.log2(i + 2) for i, p in enumerate(predicted[:k]) if p in actual)
    return dcg / idcg

a_, p_ = ["b", "e", "z"], ["a", "b", "c", "d", "e"]
print(f"P@5={precision_at_k(a_, p_, 5):.3f}  R@5={recall_at_k(a_, p_, 5):.3f}  NDCG@5={ndcg_at_k(a_, p_, 5):.3f}")

P@5=0.400  R@5=0.667  NDCG@5=0.478


### Matrix Factorization

ماتریس $R$ ($m \times n$) را با ضرب دو ماتریس کوچک تقریب می‌زنیم:

$$R \approx P\,Q^\top, \qquad \hat x_{ui} = \mathbf p_u^\top \mathbf q_i = \sum_{f=1}^{d} p_{uf}\, q_{if}$$

- $\mathbf p_u \in \mathbb R^d$: embedding کاربر؛ $\mathbf q_i \in \mathbb R^d$: embedding آیتم. هر بعد یک «latent factor» است (مثلاً میزان اکشن بودن) که مدل خودش کشف می‌کند.
- پارامترها: $(m+n)\,d$ به‌جای $m \cdot n$ خانه.
- در PyTorch: `nn.Embedding(num_users, d)` فقط یک جدول lookup قابل train است؛ سطر $u$ = $\mathbf p_u$.

### BPR — Bayesian Personalized Ranking

**ایده:** لازم نیست score دقیق باشد؛ کافی است آیتمی که کاربر با آن تعامل داشته **بالاتر** از آیتمی که نداشته قرار بگیرد.

داده‌ی train سه‌تایی است: $(u, i, j)$ با $i$ = positive و $j$ = negative (نمونه‌ی تصادفی).

$$L_{BPR} = -\sum_{(u,i,j)} \ln \sigma(\hat x_{ui} - \hat x_{uj}) \;+\; \lambda \lVert\Theta\rVert^2$$

- اگر $\hat x_{ui} \gg \hat x_{uj}$: $\sigma \to 1$، loss $\to 0$ ✓
- اگر $\hat x_{ui} < \hat x_{uj}$: $\sigma < 0.5$، loss بزرگ → gradient دو آیتم را از هم دور می‌کند.
- این یک loss **pairwise** است (در برابر pointwise مثل MSE). عملاً AUC را بهینه می‌کند.

```
BPR_TRAIN(interactions, d, lr, epochs):
    initialise P (m x d) and Q (n x d) with small random values
    repeat for each epoch:
        for each observed (u, i):                       # in mini-batches
            j <- random item (ideally one u has NOT interacted with)
            x_uij <- p_u . q_i  -  p_u . q_j
            loss  <- -ln sigmoid(x_uij)
            g <- 1 - sigmoid(x_uij)                     # = sigmoid(-x_uij): large when the order is wrong
            p_u <- p_u + lr * ( g * (q_i - q_j) - lambda * p_u )
            q_i <- q_i + lr * ( g * p_u         - lambda * q_i )
            q_j <- q_j + lr * (-g * p_u         - lambda * q_j )
    return P, Q

RECOMMEND(u, N):  scores <- Q . p_u ;  mask items u already has ;  return top-N
```

در کد استاد update ها دستی نوشته نشده؛ `loss.backward()` + Adam همین gradient ها را حساب می‌کند. یک قدم دستی برای این‌که ببینی چه می‌شود:

In [28]:
sigmoid = lambda z: 1 / (1 + np.exp(-z))
p_u, q_i, q_j, lr = np.array([1.0, 0.0]), np.array([0.5, 0.5]), np.array([1.0, 1.0]), 0.1   # i = positive, j = negative

x_uij = p_u @ q_i - p_u @ q_j
print(f"before: x_ui={p_u@q_i:.2f}  x_uj={p_u@q_j:.2f}  x_uij={x_uij:.2f}  loss={-np.log(sigmoid(x_uij)):.3f}   (negative is ranked ABOVE positive)")
g = 1 - sigmoid(x_uij)
p_new, qi_new, qj_new = p_u + lr * g * (q_i - q_j), q_i + lr * g * p_u, q_j - lr * g * p_u
x_new = p_new @ qi_new - p_new @ qj_new
print(f"g = 1 - sigmoid(x_uij) = {g:.3f}")
print(f"after : p_u={p_new.round(3)}  q_i={qi_new.round(3)}  q_j={qj_new.round(3)}")
print(f"        x_uij={x_new:.3f}  loss={-np.log(sigmoid(x_new)):.3f}   (gap moved in the right direction)")

before: x_ui=0.50  x_uj=1.00  x_uij=-0.50  loss=0.974   (negative is ranked ABOVE positive)
g = 1 - sigmoid(x_uij) = 0.622
after : p_u=[ 0.969 -0.031]  q_i=[0.562 0.5  ]  q_j=[0.938 1.   ]
        x_uij=-0.348  loss=0.882   (gap moved in the right direction)


### LightGCN

**دید گرافی:** تعامل‌ها یک **گراف دوبخشی** (bipartite) می‌سازند: user ها یک طرف، item ها طرف دیگر، هر تعامل یک یال (undirected).

ماتریس مجاورت کل گراف، با $(m+n)$ گره:
$$A = \begin{bmatrix} 0 & R \\ R^\top & 0 \end{bmatrix}$$

**ایراد MF از این دید:** embedding کاربر فقط از loss روی یال‌های خودش یاد گرفته می‌شود. **ایده‌ی GNN:** embedding هر گره را با embedding همسایه‌هایش ترکیب کن (**message passing**).

- بعد از ۱ لایه: کاربر ← آیتم‌هایی که دیده
- بعد از ۲ لایه: کاربر ← کاربرانی که همان آیتم‌ها را دیده‌اند (**همان سیگنال collaborative!**)
- بعد از ۳ لایه: کاربر ← آیتم‌هایی که آن کاربرانِ مشابه دیده‌اند

پس LightGCN در واقع **ایده‌ی kNN (همسایگی) را داخل یادگیری embedding (MF) می‌برد.** این پل بین Part 2 و Part 3 است.

**قانون propagation** (بدون weight matrix، بدون activation — برای همین «Light»):

$$\mathbf e_u^{(k+1)} = \sum_{i \in N_u} \frac{1}{\sqrt{|N_u|}\sqrt{|N_i|}}\; \mathbf e_i^{(k)} \qquad \mathbf e_i^{(k+1)} = \sum_{u \in N_i} \frac{1}{\sqrt{|N_i|}\sqrt{|N_u|}}\; \mathbf e_u^{(k)}$$

به شکل ماتریسی: $E^{(k+1)} = \hat A\, E^{(k)}$ با $\hat A = D^{-1/2} A D^{-1/2}$.

**Embedding نهایی** = میانگین همه‌ی لایه‌ها (شامل لایه‌ی 0):
$$\mathbf e_u = \frac{1}{K+1}\sum_{k=0}^{K} \mathbf e_u^{(k)}, \qquad \hat x_{ui} = \mathbf e_u^\top \mathbf e_i$$

```
LIGHTGCN_FORWARD(E0, A_hat, K):          # E0: (m+n) x d  -- the ONLY trainable parameters
    E <- E0 ;  layers <- [E0]
    for k = 1..K:
        E <- A_hat @ E                   # sparse matrix product = one round of message passing
        layers.append(E)
    E_final <- mean(layers)
    split E_final into user embeddings and item embeddings
    score(u, i) <- e_u . e_i             # then the same BPR loss as MF
```

**چرا normalization:** بدون آن، گره‌های پرارتباط (hub) با جمع زدن همسایه‌های زیاد embedding خیلی بزرگ می‌گیرند و train ناپایدار می‌شود. $\frac{1}{\sqrt{\deg(i)\deg(j)}}$ پیام روی هر یال را با درجه‌ی **هر دو** سر یال مقیاس می‌کند. «Symmetric» چون گراف undirected است.

**مثال دستی (گراف ۴ گره‌ای استاد):** درجه‌ها $[3, 2, 2, 1]$.
- یال (0,1): $1/\sqrt{3\cdot2} = 0.408$
- یال (1,2): $1/\sqrt{2\cdot2} = 0.500$
- یال (0,3): $1/\sqrt{3\cdot1} = 0.577$

هر چه دو سر یال پرارتباط‌تر، وزن پیام کمتر.

**چرا میانگین لایه‌ها:** (۱) جلوی **over-smoothing** را می‌گیرد (با لایه‌های زیاد embedding همه‌ی گره‌ها شبیه هم می‌شود)، (۲) لایه‌ی 0 (هویت خود گره) را حفظ می‌کند، (۳) سیگنال همسایگی‌های با فاصله‌ی مختلف را ترکیب می‌کند.

**تفاوت با MF:** پارامترها دقیقاً یکی است ($E^{(0)}$، یعنی $(m+n)d$). تنها فرق: قبل از ضرب داخلی، embedding ها روی گراف smooth می‌شوند. با $K=0$ لایه، LightGCN **دقیقاً همان MF** است.

**Complexity:** هر forward $O(K\cdot|E|\cdot d)$ (ضرب sparse)، در برابر $O(B\cdot d)$ برای یک batch در MF. برای همین در خروجی استاد هر epoch LightGCN کندتر است.

In [29]:
# message passing روی یک گراف کوچک: 2 user (u0,u1) و 3 item (i0,i1,i2)
R_toy = np.array([[1, 1, 0],      # u0 -> i0, i1
                  [0, 1, 1]])     # u1 -> i1, i2
m_, n_ = R_toy.shape
A = np.block([[np.zeros((m_, m_)), R_toy], [R_toy.T, np.zeros((n_, n_))]])
deg = A.sum(1)
A_hat = np.diag(deg ** -0.5) @ A @ np.diag(deg ** -0.5)
names = ["u0", "u1", "i0", "i1", "i2"]
print("degrees:", dict(zip(names, deg.astype(int).tolist())))
print(pd.DataFrame(A_hat.round(3), index=names, columns=names), "\n")

E0 = np.eye(5)                                   # one-hot "embeddings" so that we can SEE where information flows
E1, E2 = A_hat @ E0, A_hat @ A_hat @ E0
print("u0 after 1 layer :", dict(zip(names, E1[0].round(3).tolist())), " <- only its items")
print("u0 after 2 layers:", dict(zip(names, E2[0].round(3).tolist())), " <- now it sees u1 (shares item i1)")

degrees: {'u0': 2, 'u1': 2, 'i0': 1, 'i1': 2, 'i2': 1}
       u0     u1     i0   i1     i2
u0  0.000  0.000  0.707  0.5  0.000
u1  0.000  0.000  0.000  0.5  0.707
i0  0.707  0.000  0.000  0.0  0.000
i1  0.500  0.500  0.000  0.0  0.000
i2  0.000  0.707  0.000  0.0  0.000 

u0 after 1 layer : {'u0': 0.0, 'u1': 0.0, 'i0': 0.707, 'i1': 0.5, 'i2': 0.0}  <- only its items
u0 after 2 layers: {'u0': 0.75, 'u1': 0.25, 'i0': 0.0, 'i1': 0.0, 'i2': 0.0}  <- now it sees u1 (shares item i1)


## 3.C Professor's code — چه کار می‌کند

**Pipeline نوت‌بوک `gnn.ipynb`:**

1. **داده:** Gowalla → فقط ستون‌های (user, item) → فیلتر **50-core** (کاربر و آیتم با حداقل ۵۰ تعامل، برای سرعت) → map کردن ID ها به اندیس پیوسته `0..N-1` (لازم برای `nn.Embedding`).
2. **Split:** برای هر کاربر 80٪ train / 20٪ test، تصادفی.
3. **`R`:** ماتریس sparse دودویی train (user × item).
4. **Metrics:** سه تابع بالا + `evaluate_model`: embedding ها را **یک بار** می‌گیرد؛ برای هر کاربر `scores = u_emb @ items_emb.T`؛ آیتم‌های train را `-inf` می‌کند؛ `torch.topk`.
5. **`MatrixFactorization`:** دو `nn.Embedding`؛ `forward(users, pos, neg)` → `(u*pos).sum(1)` و `(u*neg).sum(1)`.
6. **`LightGCN`:** همان دو embedding + `_build_norm_adj` ($\hat A$ را می‌سازد) + حلقه‌ی `torch.sparse.mm` + `torch.mean` روی لایه‌ها.
7. **`BPRDataset`:** هر نمونه `(user, pos_item, neg_item)` با `neg_item = randint(num_items)`.
8. **Train:** Adam، `loss = -F.logsigmoid(pos_scores - neg_scores).mean()`.

**خط‌های کلیدی LightGCN استاد:**

```python
# A = [[0, R], [R^T, 0]]
adj_mat[:num_users, num_users:] = R
adj_mat[num_users:, :num_users] = R.T
# D^{-1/2}
rowsum = np.array(adj_mat.sum(axis=1))
d_inv  = np.power(rowsum, -0.5).flatten()
d_inv[np.isinf(d_inv)] = 0.                        # isolated nodes: degree 0 -> inf -> set to 0
norm_adj = d_mat.dot(adj_mat).dot(d_mat)           # D^{-1/2} A D^{-1/2}

# forward
all_emb = torch.cat([self.user_emb.weight, self.item_emb.weight])   # E^(0): users stacked on items
embs = [all_emb]
for layer in range(self.n_layers):
    all_emb = torch.sparse.mm(norm_adj, all_emb)   # E^(k+1) = A_hat E^(k)
    embs.append(all_emb)
light_out = torch.mean(torch.stack(embs, dim=1), dim=1)             # average over layers
users_emb, items_emb = torch.split(light_out, [num_users, num_items])
```

**ایرادها و نکته‌ها:**

1. **Leakage / تکرار در داده:** Gowalla check-in است و یک کاربر می‌تواند چند بار به یک مکان برود. split روی check-in ها انجام شده، نه روی جفت‌های یکتا؛ در خروجی استاد هم پیداست (`User 0: [32, 27, 1, 1, 46]`). پس یک جفت (user, item) می‌تواند هم در train باشد هم در test. و چون در evaluation آیتم‌های train با `-inf` mask می‌شوند، آن آیتم‌های test **هرگز** قابل hit نیستند → recall مصنوعاً پایین می‌آید. اصلاح: قبل از split، `drop_duplicates(['user','item'])`.
2. **Negative sampling ساده:** `randint` چک نمی‌کند آیتم واقعاً منفی باشد (false negative ممکن است). تقریب رایج و ارزان، چون داده خیلی sparse است.
3. **بدون L2 regularization** — در مقاله‌ی LightGCN و BPR ترم $\lambda\lVert\Theta\rVert^2$ هست.
4. **منطق propagation دو بار کپی شده** (`forward` و `get_embeddings`)؛ اسم `g_droped_adj` از کد اصلی مانده ولی dropout ای در کار نیست.
5. **ساخت adjacency با `dok_matrix` و حلقه** کند است؛ `scipy.sparse.bmat` یک خط است.
6. **فقط ۱۰ epoch:** هر دو مدل under-train اند (Recall@20 برای MF ≈ 0.09)، پس مقایسه‌ی نهایی آن‌جا دقیق نیست.

## 3.D اجرا روی MovieLens (ادامه‌ی همان داده‌ی Part 1 و 2)

به‌جای Gowalla (100MB) همان MovieLens را implicit می‌کنیم: `rating >= 4` یعنی positive. این‌طوری سه بخش نوت‌بوک روی یک داده قابل مقایسه‌اند. split مثل استاد: 80/20 برای هر کاربر (روی جفت‌های یکتا).

In [30]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(SEED)

pos_df = ratings[ratings.rating >= 4.0]
rng = np.random.default_rng(SEED)
train_pairs, test_data = [], {}
for u, g in pos_df.groupby("userId"):
    items = rng.permutation(i2idx[g.movieId].to_numpy())
    cut = int(0.8 * len(items)) if len(items) >= 5 else len(items)
    train_pairs += [(int(u2idx[u]), int(i)) for i in items[:cut]]
    if cut < len(items): test_data[int(u2idx[u])] = items[cut:].tolist()
train_pairs = np.array(train_pairs)
R = csr_matrix((np.ones(len(train_pairs)), (train_pairs[:, 0], train_pairs[:, 1])), shape=(n_users, n_items))
print(f"train interactions: {len(train_pairs)}   test users: {len(test_data)}")

def evaluate_scores(scores, k=20):
    """scores: users x items. Same logic as the professor's evaluate_model: mask train items, take top-k, average the metrics."""
    sc = np.array(scores, dtype=float)
    sc[R.nonzero()] = -np.inf
    top = np.argsort(-sc, axis=1)[:, :k]
    res = [(precision_at_k(a, top[u].tolist(), k), recall_at_k(a, top[u].tolist(), k), ndcg_at_k(set(a), top[u].tolist(), k)) for u, a in test_data.items()]
    return dict(zip([f"Precision@{k}", f"Recall@{k}", f"NDCG@{k}"], np.mean(res, axis=0)))

results3 = {"Popularity": evaluate_scores(np.tile(np.asarray(R.sum(0)).ravel(), (n_users, 1)))}
S3 = topk_item_sim(R, k=30, metric="cosine")
results3["item-kNN cosine (Part 2)"] = evaluate_scores((R @ S3).toarray())
pd.DataFrame(results3).T

train interactions: 38620   test users: 603


,Precision@20,Recall@20,NDCG@20
Popularity,0.0866,0.1450,0.1520
item-kNN cosine (Part 2),0.1191,0.2052,0.2075


### MF و LightGCN در PyTorch (ساختار استاد، فشرده‌تر)

تغییرها نسبت به کد استاد: propagation فقط یک جا نوشته شده (`get_embeddings`) و LightGCN از MF ارث می‌برد — که خودش نشان می‌دهد **تنها تفاوت دو مدل همین یک تابع است**.

In [31]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embedding_dim)    # P: one trainable row per user
        self.item_emb = nn.Embedding(num_items, embedding_dim)    # Q: one trainable row per item
        nn.init.normal_(self.user_emb.weight, std=0.1)
        nn.init.normal_(self.item_emb.weight, std=0.1)

    def get_embeddings(self):
        return self.user_emb.weight, self.item_emb.weight

    def forward(self, users, pos_items, neg_items):
        U, I = self.get_embeddings()
        u = U[users]
        return (u * I[pos_items]).sum(1), (u * I[neg_items]).sum(1)   # x_ui, x_uj


def build_norm_adj(R):
    """A_hat = D^-1/2 [[0, R], [R^T, 0]] D^-1/2  as a sparse torch tensor."""
    A = sp.bmat([[None, R], [R.T, None]], format="csr")
    deg = np.asarray(A.sum(1)).ravel()
    with np.errstate(divide="ignore"):
        d_inv = np.power(deg, -0.5)
    d_inv[np.isinf(d_inv)] = 0.0                                  # isolated nodes
    A = (sp.diags(d_inv) @ A @ sp.diags(d_inv)).tocoo()
    return torch.sparse_coo_tensor(np.vstack([A.row, A.col]), A.data.astype(np.float32), A.shape).coalesce()


class LightGCN(MatrixFactorization):
    def __init__(self, num_users, num_items, R, embedding_dim=64, n_layers=2):
        super().__init__(num_users, num_items, embedding_dim)
        self.num_users, self.num_items, self.n_layers = num_users, num_items, n_layers
        self.norm_adj = build_norm_adj(R)

    def get_embeddings(self):
        E = torch.cat([self.user_emb.weight, self.item_emb.weight])   # E^(0)
        layers = [E]
        for _ in range(self.n_layers):
            E = torch.sparse.mm(self.norm_adj, E)                     # message passing
            layers.append(E)
        E = torch.stack(layers, dim=1).mean(dim=1)                    # layer combination
        return torch.split(E, [self.num_users, self.num_items])


def train_bpr(model, epochs=60, lr=5e-3, batch_size=2048):
    torch.manual_seed(SEED)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    U, I = torch.tensor(train_pairs[:, 0]), torch.tensor(train_pairs[:, 1])
    t0 = time.time()
    for ep in range(epochs):
        perm = torch.randperm(len(U))
        neg  = torch.randint(0, n_items, (len(U),))                   # negative sampling: fresh negatives every epoch
        tot  = 0.0
        for s in range(0, len(U), batch_size):
            b = perm[s:s + batch_size]
            pos_scores, neg_scores = model(U[b], I[b], neg[b])
            loss = -F.logsigmoid(pos_scores - neg_scores).mean()      # BPR loss
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(b)
        if (ep + 1) % 20 == 0:
            print(f"  {model.__class__.__name__:20s} epoch {ep+1:3d}  loss={tot/len(U):.4f}  ({time.time()-t0:.0f}s)")
    with torch.no_grad():
        Ue, Ie = model.get_embeddings()
        return (Ue @ Ie.T).numpy()

results3["MF + BPR (PyTorch)"]       = evaluate_scores(train_bpr(MatrixFactorization(n_users, n_items)))
results3["LightGCN + BPR (PyTorch)"] = evaluate_scores(train_bpr(LightGCN(n_users, n_items, R, n_layers=2)))
pd.DataFrame(results3).T

  MatrixFactorization  epoch  20  loss=0.0608  (2s)


  MatrixFactorization  epoch  40  loss=0.0393  (3s)


  MatrixFactorization  epoch  60  loss=0.0339  (5s)


  LightGCN             epoch  20  loss=0.0927  (10s)


  LightGCN             epoch  40  loss=0.0592  (20s)


  LightGCN             epoch  60  loss=0.0448  (30s)


,Precision@20,Recall@20,NDCG@20
Popularity,0.0866,0.1450,0.1520
item-kNN cosine (Part 2),0.1191,0.2052,0.2075
MF + BPR (PyTorch),0.1010,0.1860,0.1700
LightGCN + BPR (PyTorch),0.1290,0.2251,0.2219


**خواندن جدول:** با پارامترهای یکسان و loss یکسان، فقط اضافه کردن message passing به MF عدد را بالا می‌برد؛ و هر دو مدلِ یادگرفته از popularity بهترند. هزینه‌اش زمان train بیشتر است (ستون ثانیه‌ها را در log مقایسه کن).

item-kNN ساده را هم ببین: یک روش بدون training که کاملاً رقابتی است. این نتیجه در ادبیات هم شناخته‌شده است و دلیل این‌که در صنعت همیشه baseline های ساده‌ی **خوب‌tune‌شده** را کنار مدل deep می‌گذارند.

## 3.E Industry version

### `implicit`: ALS و BPR

- **ALS (Alternating Least Squares)** — رایج‌ترین MF برای implicit data در صنعت (Spark MLlib هم همین را دارد). به‌جای SGD، یک بار $P$ را ثابت می‌گیرد و $Q$ را با least squares حل می‌کند و برعکس؛ به‌راحتی parallel می‌شود و negative sampling لازم ندارد (همه‌ی خانه‌های خالی را با وزن کم منفی حساب می‌کند).
- **BPR** — همان الگوریتم بالا، پیاده‌سازی C++.

In [32]:
from implicit.als import AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking

als = AlternatingLeastSquares(factors=64, regularization=0.05, iterations=20, random_state=SEED)
als.fit(R, show_progress=False)
results3["implicit ALS"] = evaluate_scores(als.user_factors @ als.item_factors.T)

bpr = BayesianPersonalizedRanking(factors=64, learning_rate=0.01, regularization=0.01, iterations=100, random_state=SEED)   # library defaults
bpr.fit(R, show_progress=False)
results3["implicit BPR"] = evaluate_scores(bpr.user_factors @ bpr.item_factors.T)
pd.DataFrame(results3).T

,Precision@20,Recall@20,NDCG@20
Popularity,0.0866,0.1450,0.1520
item-kNN cosine (Part 2),0.1191,0.2052,0.2075
MF + BPR (PyTorch),0.1010,0.1860,0.1700
LightGCN + BPR (PyTorch),0.1290,0.2251,0.2219
implicit ALS,0.1150,0.2278,0.2216
implicit BPR,0.1044,0.2002,0.1926


عددها به hyperparameter حساس‌اند (factors، learning rate، regularization، تعداد iteration) و این‌جا tune نشده‌اند. نمونه: وقتی برای `implicit` BPR تعداد iteration را بالا بردم و regularization را کم کردم، overfit شد و Recall@20 از حدود 0.20 به زیر popularity افتاد. در کار واقعی این‌ها را روی یک validation set جدا tune می‌کنی، نه روی test.

### LightGCN با PyTorch Geometric

PyG کتابخانه‌ی استاندارد GNN است و `LightGCN` آماده دارد (شامل BPR loss با regularization). گره‌ها یک فضای اندیس مشترک دارند: user ها `0..m-1`، item ها `m..m+n-1`.

In [33]:
try:
    from torch_geometric.nn import LightGCN as PyGLightGCN

    torch.manual_seed(SEED)
    ei = torch.tensor(np.vstack([train_pairs[:, 0], train_pairs[:, 1] + n_users]))    # user -> item edges
    edge_index = torch.cat([ei, ei.flip(0)], dim=1)                                    # undirected: add both directions
    pyg = PyGLightGCN(num_nodes=n_users + n_items, embedding_dim=64, num_layers=2)
    opt = torch.optim.Adam(pyg.parameters(), lr=5e-3)

    for ep in range(60):
        perm = torch.randperm(ei.shape[1])
        neg  = torch.randint(n_users, n_users + n_items, (ei.shape[1],))
        for s in range(0, ei.shape[1], 2048):
            b = perm[s:s + 2048]
            pairs = torch.cat([ei[:, b], torch.stack([ei[0, b], neg[b]])], dim=1)      # positive pairs, then negative pairs
            pos_rank, neg_rank = pyg(edge_index, pairs).chunk(2)
            loss = pyg.recommendation_loss(pos_rank, neg_rank, node_id=pairs.unique(), lambda_reg=1e-4)   # BPR + L2
            opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        E = pyg.get_embedding(edge_index)
        results3["LightGCN (PyTorch Geometric)"] = evaluate_scores((E[:n_users] @ E[n_users:].T).numpy())
except ImportError:
    print("torch_geometric is not installed ->  %pip install torch_geometric")
pd.DataFrame(results3).T

,Precision@20,Recall@20,NDCG@20
Popularity,0.0866,0.1450,0.1520
item-kNN cosine (Part 2),0.1191,0.2052,0.2075
MF + BPR (PyTorch),0.1010,0.1860,0.1700
LightGCN + BPR (PyTorch),0.1290,0.2251,0.2219
implicit ALS,0.1150,0.2278,0.2216
implicit BPR,0.1044,0.2002,0.1926
LightGCN (PyTorch Geometric),0.1250,0.2184,0.2159


### ابزارهای دیگر که باید اسم‌شان را بدانی

| ابزار | کاربرد |
|---|---|
| **RecBole** | ده‌ها مدل (BPR، LightGCN، …) با config، برای مقایسه‌ی منصفانه و سریع؛ بیشتر پژوهشی |
| **Spark MLlib ALS** | MF روی داده‌ی distributed |
| **TorchRec** (Meta)، **TensorFlow Recommenders** (Google)، **NVIDIA Merlin** | مدل‌های deep در scale بالا |
| **FAISS / ScaNN** | serving: پیدا کردن نزدیک‌ترین item embedding ها به user embedding |

**معماری رایج production:** embedding ها offline train می‌شوند (MF / two-tower / GNN) → item embedding ها در ANN index → online: user embedding → چند صد candidate از index → یک ranker سنگین‌تر → business rules → نمایش.

## 3.F Exam — MF / BPR / LightGCN

> در `gnn.ipynb` سوال تمرینی نیست؛ این‌ها پیش‌بینی من از سوال‌های محتمل‌اند.

**C1. What is the difference between explicit and implicit feedback, and how does it change the evaluation?**
Explicit feedback is a stated preference (ratings); implicit feedback is observed behaviour (clicks, check-ins, purchases) — only positives are observed and a missing entry is unknown, not negative. With implicit data rating-error metrics are meaningless, so ranking metrics (Precision@K, Recall@K, NDCG@K, Hit Ratio) are used, and training needs negative sampling.

**C2 (calculation). predicted = [a, b, c, d, e], actual = {b, e, z}. Compute Precision@5, Recall@5, NDCG@5.** → 3.B: **0.4, 0.667, 0.478**.

**C3. Why use NDCG in addition to Precision/Recall?**
Precision@K and Recall@K ignore the order inside the top-K. NDCG discounts each hit by $1/\log_2(\text{position}+1)$, so relevant items near the top count more; dividing by the ideal DCG normalises it to [0, 1].

**C4. Explain Matrix Factorization for recommendation.**
Each user and item is represented by a $d$-dimensional latent vector; the predicted preference is their dot product, $\hat x_{ui}=\mathbf p_u^\top\mathbf q_i$. The vectors are learned by minimising a loss on observed interactions. It compresses the $m\times n$ matrix into $(m+n)d$ parameters and generalises to pairs with no co-occurrence.

**C5. Write the BPR loss and explain the intuition. What is a training sample?**
$L=-\sum_{(u,i,j)}\ln\sigma(\hat x_{ui}-\hat x_{uj})$. A sample is a triple (user, positive item, sampled negative item). The loss is small when the positive is scored higher than the negative, so the model learns a correct *relative order* rather than absolute scores (pairwise ranking loss).

**C6 (calculation). $\hat x_{ui}=2.0$, $\hat x_{uj}=0.5$. Compute the BPR loss for this triple.**
$\sigma(1.5)=0.8176$ → loss $=-\ln 0.8176 = 0.201$. (If the scores were swapped: $\sigma(-1.5)=0.1824$ → loss $=1.70$.)

**C7. Describe LightGCN's propagation rule. How does it differ from MF and from a standard GCN?**
$E^{(k+1)}=D^{-1/2}AD^{-1/2}E^{(k)}$ on the user–item bipartite graph; the final embedding is the mean of the embeddings from all layers; the score is a dot product trained with BPR. Versus a standard GCN: no feature transformation matrices and no non-linear activation. Versus MF: same trainable parameters (only the layer-0 embeddings), but embeddings are smoothed over the graph so higher-order collaborative signals are encoded; with 0 layers it reduces to MF.

**C8. Why is the adjacency matrix symmetrically normalised? Compute the weight of an edge between nodes of degree 3 and 2.**
Without normalisation, high-degree nodes accumulate very large embeddings, which causes numerical instability; the factor $1/\sqrt{\deg(i)\deg(j)}$ keeps the scale consistent. Weight $=1/\sqrt{6}=0.408$.

**C9. What information does a user embedding contain after 1 / 2 / 3 layers?**
1: the items the user interacted with. 2: other users who share those items. 3: the items of those similar users (candidate recommendations).

**C10. What is over-smoothing and how does LightGCN mitigate it?**
With many layers, repeated neighbourhood averaging makes all node embeddings similar and they lose discriminative power. LightGCN uses few layers and averages the embeddings of all layers (including layer 0).

**C11. Why are training items masked (set to −∞) during evaluation?**
The model is trained to score them highly, so they would fill the top-K; but recommending already-seen items is useless and they are not in the test set, so they must be excluded.

**C12. Why is negative sampling needed and what is its weakness in the notebook?**
Implicit data has only positives; BPR needs an item assumed to be less preferred. The notebook samples a uniformly random item without checking whether the user interacted with it, so some "negatives" are actually positives (false negatives), and uniform negatives are mostly easy.

**C13. Build the adjacency matrix $A$ for $R=\begin{bmatrix}1&1&0\\0&1&1\end{bmatrix}$ and give the node degrees.**
$A=\begin{bmatrix}0&R\\R^\top&0\end{bmatrix}$ is $5\times5$; degrees: $u_0=2,\ u_1=2,\ i_0=1,\ i_1=2,\ i_2=1$. E.g. $\hat A_{u_0,i_0}=1/\sqrt{2\cdot1}=0.707$, $\hat A_{u_0,i_1}=1/\sqrt{2\cdot2}=0.5$.

---
# 4. جمع‌بندی و مرور شب امتحان

### نقشه‌ی کل هفته در یک جدول

| | Baselines | Item-kNN CF | MF + BPR | LightGCN |
|---|---|---|---|---|
| شخصی‌سازی | ✗ | ✓ | ✓ | ✓ |
| Training | ندارد | ندارد (precompute) | SGD / ALS | SGD |
| چه چیزی ذخیره می‌شود | $O(n)$ عدد | $O(nk)$ همسایه | $(m+n)d$ | $(m+n)d$ + گراف |
| Offline cost | $O(\lvert R\rvert)$ | $O(n^2\bar u)$ | $O(\text{epochs}\cdot\lvert R\rvert d)$ | $O(\text{steps}\cdot K\lvert E\rvert d)$ |
| Online cost / request | $O(1)$ | $O(\lvert I_u\rvert k)$ | $O(nd)$ یا ANN | $O(nd)$ یا ANN |
| توضیح‌پذیری | بالا | بالا | کم | کم |
| ضعف اصلی | همه یک لیست | sparsity، $n^2$ | فقط تعامل مستقیم | هزینه‌ی train، over-smoothing |
| کتابخانه | pandas، Surprise | `implicit`، Surprise، FAISS | `implicit`، Spark ALS | PyG، RecBole |

### فرمول‌هایی که باید حفظ باشی

- MAE $=\frac1{|T|}\sum|r-\hat r|$ ، RMSE $=\sqrt{\frac1{|T|}\sum(r-\hat r)^2}$ ، Hit Ratio $=\frac{\#hits}{|T|}$
- Jaccard $=\frac{|A\cap B|}{|A\cup B|}$ ، Cosine $=\frac{\mathbf u\cdot\mathbf v}{\lVert\mathbf u\rVert\lVert\mathbf v\rVert}$ ، Adjusted cosine = cosine بعد از کم کردن $\bar r_u$
- Item-kNN: $\hat r_{ui}=\frac{\sum_j s_{ij}r_{uj}}{\sum_j|s_{ij}|}$
- Precision@K $=\frac{hits}{K}$ ، Recall@K $=\frac{hits}{|actual|}$ ، NDCG $=\frac{DCG}{IDCG}$ با $\frac{1}{\log_2(pos+1)}$
- MF: $\hat x_{ui}=\mathbf p_u^\top\mathbf q_i$ ، BPR: $-\ln\sigma(\hat x_{ui}-\hat x_{uj})$
- LightGCN: $E^{(k+1)}=D^{-1/2}AD^{-1/2}E^{(k)}$ ، $E=\text{mean}_k E^{(k)}$

### عددهای مرجع

- Random HR@K با 1+99 $= K/100$
- $1/\log_2$: رتبه 1 → 1، رتبه 2 → 0.631، رتبه 3 → 0.5، رتبه 4 → 0.431، رتبه 5 → 0.387
- $\sigma(0)=0.5$ ، $-\ln 0.5 = 0.693$ (BPR loss در شروع train، وقتی مدل هنوز چیزی نمی‌داند)
- kNN-map: زمان $O(n^2)$ ، حافظه $O(nk)$

### اشتباه‌های رایج در امتحان

1. در cosine دستی، بردارها را فقط روی **کاربران مشترک** بگیر (طبق روش استاد)، ولی در Jaccard اجتماع **همه‌ی** کاربران هر دو آیتم است.
2. مخرج prediction **جمع شباهت‌ها** است، نه تعداد همسایه‌ها.
3. مخرج Recall تعداد آیتم‌های **actual** است؛ مخرج Precision خود **K**.
4. IDCG را با $\min(K, |actual|)$ جمله حساب کن.
5. MAE پایین‌تر بهتر است؛ HR/Precision/Recall/NDCG بالاتر بهتر.
6. «Jaccard مقدار rating را نادیده می‌گیرد» ≠ «Jaccard بدتر است» — در نوت‌بوک استاد بهتر شد و باید بتوانی دلیلش را بگویی.

### برای این‌که الگوریتم‌ها واقعاً جا بیفتند (پیشنهاد تمرین)

- روی ماتریس Q4، **بدون نگاه کردن**، Jaccard و cosine همه‌ی جفت‌ها را حساب کن و با تابع‌های بخش 2.B چک کن.
- pseudocode این پنج الگوریتم را از حفظ بنویس: HR evaluation، `PREDICT`، `TOPN_KNN_MAP`، Amazon item-to-item، `LIGHTGCN_FORWARD`.
- در کد 2.D مقدار `k` را عوض کن (5، 30، 200) و اثرش را روی MAE و HR ببین.
- در `LightGCN` مقدار `n_layers` را 0، 1، 3 بگذار؛ با 0 باید همان MF دربیاید.